# The Annotated Transformer, rebuilt for an RTX 4090

This notebook rebuilds **"Attention Is All You Need"** (Vaswani et al., 2017, [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)) in code you can run and trace one cell at a time. It follows Harvard NLP's [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) (Sasha Rush, 2018; updated 2022 by Austin Huang, Suraj Subramanian, Jonathan Sum, Khalid Almubarak and Stella Biderman), and adds what that post leaves out or hands to libraries:

- a **byte-pair-encoding tokenizer written from scratch**, then the fast library version the model actually uses
- **tests that pin each component to a known-correct reference** (PyTorch's own attention, `nn.MultiheadAttention`, `nn.LayerNorm`, causality and padding invariance)
- **label smoothing** two ways, the **warmup learning-rate schedule**, **bf16 mixed precision** and **token-based batching**
- warm-up runs on **synthetic copy and reverse tasks**, then **German→English translation on Multi30k**
- **greedy decoding** and **beam search with the paper's length penalty**, plus **checkpoint averaging**
- **BLEU written from scratch** and checked against sacreBLEU, plus chrF, perplexity and accuracy
- **attention maps** for every layer and head, and small **ablations** in the style of the paper's Table 3

### How to use it

1. Run the cells in order. Each part goes **idea → code → check**: a check is an `assert`, a printed shape or a plot. If a check surprises you, stop and work out why before moving on.
2. Shapes are printed throughout. The notation used everywhere:

| Symbol | Meaning |
|---|---|
| `B` | batch size (sentences) |
| `S`, `T` | source length, target length (in tokens) |
| `d_model` | width of every vector in the model (paper: 512) |
| `h` | number of attention heads (paper: 8) |
| `d_k = d_v = d_model / h` | width of one head (paper: 64) |
| `d_ff` | hidden width of the feed-forward block (paper: 2048) |
| `N` | number of encoder layers = number of decoder layers (paper: 6) |
| `V` | vocabulary size |

3. **Quick mode.** Set the environment variable `TRANSFORMER_NB_QUICK=1` before starting Jupyter to run everything with a tiny model and a slice of the data. It finishes in a few minutes even on a CPU, which is useful as a smoke test. Leave it unset on the 4090 for the real run. The long cells print their own timings.

4. Run it on the Jupyter kernel of your **`train`** environment (the PyTorch one from your Day 1 setup). It needs `torch` (the CUDA build on the 4090), `tokenizers`, `sacrebleu`, `matplotlib` and `numpy`; the first code cell shows the one-line install if anything is missing.

5. **On a Mac with Apple Silicon** it runs on the Mac's GPU through PyTorch's MPS backend, in float32. Everything works the same; training is just much slower than on the 4090. A model trained on the 4090 can be reused there: copy the `checkpoints/` folder across and the training cell loads it instead of retraining.

### Contents

| Part | Topic | Paper section |
|---|---|---|
| 0 | Setup | — |
| 1 | Data: Multi30k | 5.1 |
| 2 | Tokenization: BPE from scratch, then a shared BPE vocabulary | 5.1 |
| 3 | Batches, teacher forcing and masks | 3.2.3, 5.1 |
| 4 | The model: embeddings, positional encoding, attention, multi-head attention, feed-forward, residuals + LayerNorm, encoder, decoder | 3.1–3.5 |
| 5 | Training machinery: label smoothing, Adam + warmup schedule, the training loop | 5.3, 5.4 |
| 6 | Warm-up on synthetic data: copy and reverse; greedy decoding | — |
| 7 | Training on Multi30k (German→English); checkpoint averaging | 5, 6.1 |
| 8 | Inference: greedy decoding and beam search | 6.1 |
| 9 | Evaluation: perplexity, accuracy, BLEU from scratch, sacreBLEU, chrF | 6.1 |
| 10 | Visualising attention | Appendix |
| 11 | Why self-attention: what attention costs | 4 |
| 12 | Ablations in the style of Table 3 (optional) | 6.2 |
| 13 | Recap, a map from paper to code, exercises | — |

## Part 0 · Setup

Imports, the device, mixed precision and a plotting style. The device is picked automatically: an NVIDIA GPU (CUDA) if there is one, otherwise an Apple Silicon GPU (MPS), otherwise the CPU. Two settings matter for speed on the 4090:

- **TF32** (`torch.set_float32_matmul_precision("high")`): float32 matrix multiplies run on tensor cores with a 10-bit mantissa. Free speed, no code changes.
- **bf16 autocast**: inside `with amp():` matrix multiplies run in bfloat16. bf16 has float32's exponent range, so unlike float16 it needs no loss scaling. Softmax, log-softmax and LayerNorm stay in float32 automatically.

On a Mac (MPS) or the CPU, both are skipped and everything runs in plain float32. The results are the same, just slower.

In [ ]:
# If an import fails, install the missing package into this kernel's environment, e.g.:
#   %pip install tokenizers sacrebleu matplotlib numpy
import contextlib, copy, gzip, math, os, pathlib, random, ssl, time, unicodedata, urllib.error, urllib.request
from collections import Counter
from dataclasses import dataclass, asdict, replace

# On Apple GPUs, run any operation MPS doesn't implement on the CPU instead of failing.
# (Read when torch is imported, so it has to be set first.)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import tokenizers, sacrebleu
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

QUICK = os.environ.get("TRANSFORMER_NB_QUICK", "0") == "1"   # tiny smoke-test mode (see the intro)

if torch.cuda.is_available():
    device = torch.device("cuda")                  # NVIDIA GPU (the 4090)
elif torch.backends.mps.is_available():
    device = torch.device("mps")                   # Apple Silicon GPU
else:
    device = torch.device("cpu")
torch.set_float32_matmul_precision("high")   # TF32 tensor cores for float32 matmuls (CUDA only)
AMP_DTYPE = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else None

def amp():
    # Mixed-precision context: bf16 autocast on the NVIDIA GPU, plain float32 everywhere else.
    if AMP_DTYPE is None:
        return contextlib.nullcontext()
    return torch.autocast(device_type=device.type, dtype=AMP_DTYPE)

def sync():
    # GPUs run asynchronously: wait for queued work to finish before reading a timer.
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)   # torch.manual_seed covers CUDA and MPS too

seed_everything(0)
device_name = {"cuda": torch.cuda.get_device_name(0) if device.type == "cuda" else "",
               "mps": "Apple Silicon GPU (MPS)", "cpu": "CPU"}[device.type]
print(f"torch {torch.__version__} | tokenizers {tokenizers.__version__} | sacrebleu {sacrebleu.__version__}")
print("device:", device, "|", device_name)
print("autocast dtype:", AMP_DTYPE, "| QUICK mode:", QUICK)

In [ ]:
# Plot style: one blue ramp for magnitudes (attention, masks), blue <-> red for signed values
# (positional encodings), and a fixed series order for line charts.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
SEQ_CMAP = LinearSegmentedColormap.from_list(
    "seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281", "#0d366b"])
DIV_CMAP = LinearSegmentedColormap.from_list(
    "div_blue_red", ["#104281", "#3987e5", "#cde2fb", "#f0efec", "#f4b6b5", "#e34948", "#a8302f"])
plt.rcParams.update({
    "figure.dpi": 110, "axes.prop_cycle": plt.cycler(color=SERIES),
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.0, "legend.frameon": False,
})

def describe(name, t):
    # Print a tensor's shape and dtype: the habit that makes transformer code traceable.
    print(f"{name:<34} shape={str(tuple(t.shape)):<20} dtype={str(t.dtype).replace('torch.', '')}")

## Part 1 · Data: Multi30k

The paper trains on WMT 2014: about 4.5 million English–German sentence pairs, and 36 million English–French pairs. The base model trained for 12 hours on 8 P100 GPUs, the big model for 3.5 days. To learn the mechanics we use **Multi30k** (Elliott et al., 2016) instead: image descriptions written in English and translated into German. It has 29,000 training pairs, 1,014 validation pairs and 1,000 test pairs (the 2016 Flickr test set). It's small enough that a full Transformer trains in minutes on one GPU, and real enough that the translations are worth reading.

We translate **German → English** (as The Annotated Transformer does), so you can judge the outputs at a glance. The paper's direction was English → German; swapping is a one-line change below.

The files come straight from the dataset's GitHub repository and are cached in `data/multi30k/`.

In [ ]:
DATA_DIR = pathlib.Path("data/multi30k")
BASE_URL = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw"
SPLITS = {"train": "train", "val": "val", "test": "test_2016_flickr"}
SRC_LANG, TGT_LANG = "de", "en"   # set to ("en", "de") for the paper's direction

def fetch(url, path):
    # Download to a temporary name first, so an interrupted download never leaves a broken file behind.
    try:
        response = urllib.request.urlopen(url)
    except urllib.error.URLError as err:
        if not isinstance(err.reason, ssl.SSLCertVerificationError):
            raise
        import certifi            # some macOS Pythons can't find the system certificates; use Mozilla's bundle
        response = urllib.request.urlopen(url, context=ssl.create_default_context(cafile=certifi.where()))
    tmp = path.with_name(path.name + ".part")
    with response, open(tmp, "wb") as f:
        f.write(response.read())
    tmp.rename(path)

def download_multi30k(data_dir=DATA_DIR):
    data_dir.mkdir(parents=True, exist_ok=True)
    for stem in SPLITS.values():
        for lang in ("de", "en"):
            path = data_dir / f"{stem}.{lang}.gz"
            if not path.exists():
                fetch(f"{BASE_URL}/{stem}.{lang}.gz", path)
    return sorted(p.name for p in data_dir.glob("*.gz"))

def read_lines(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def load_split(split):
    stem = SPLITS[split]
    src = read_lines(DATA_DIR / f"{stem}.{SRC_LANG}.gz")
    tgt = read_lines(DATA_DIR / f"{stem}.{TGT_LANG}.gz")
    assert len(src) == len(tgt), "source and target files must align line by line"
    return list(zip(src, tgt))

print("files:", download_multi30k())
raw = {split: load_split(split) for split in SPLITS}
if QUICK:
    raw = {"train": raw["train"][:3000], "val": raw["val"][:200], "test": raw["test"][:200]}
for split, pairs in raw.items():
    print(f"{split:>5}: {len(pairs):,} sentence pairs")

In [ ]:
for src, tgt in random.Random(1).sample(raw["train"], 4):
    print(f"{SRC_LANG.upper()}: {src}\n{TGT_LANG.upper()}: {tgt}\n")

In [ ]:
src_words = [len(s.split()) for s, _ in raw["train"]]
tgt_words = [len(t.split()) for _, t in raw["train"]]
print(f"words per sentence  {SRC_LANG}: mean {np.mean(src_words):.1f}, max {max(src_words)}   "
      f"{TGT_LANG}: mean {np.mean(tgt_words):.1f}, max {max(tgt_words)}")

fig, ax = plt.subplots(figsize=(7, 3))
bins = np.arange(0, 45)
ax.hist(src_words, bins=bins, histtype="step", linewidth=2, label=SRC_LANG)
ax.hist(tgt_words, bins=bins, histtype="step", linewidth=2, label=TGT_LANG)
ax.set_xlabel("words per sentence"); ax.set_ylabel("sentences"); ax.set_title("Multi30k training sentence lengths")
ax.legend(); plt.show()

## Part 2 · Tokenization

A model works on integers, so text has to be cut into **tokens** and each token mapped to an id. The choice of unit matters:

| Unit | Vocabulary | Sequence length | Unseen words |
|---|---|---|---|
| characters | tiny (~100) | very long: attention cost grows with length² | never a problem |
| words | huge (German compounds alone run to millions) | short | become `<unk>`: the model cannot read or write them |
| **subwords** (BPE, WordPiece) | you choose (8k–50k) | short | split into known pieces: `Schutzhelmen` → `▁Schutz` + `hel` + `men` |

The paper uses subwords: **byte-pair encoding** with a **shared source–target vocabulary of about 37,000 tokens** for English–German (and a 32,000 word-piece vocabulary for English–French). Sharing the vocabulary lets names, numbers and cognates map to the same ids in both languages, and it allows the paper's weight tying, which you will meet in Part 4.

**BPE** (Sennrich et al., 2016) is a compression algorithm repurposed for vocabularies:

1. Start with every word split into characters, with a word-start marker (`▁`, as in SentencePiece).
2. Count every adjacent pair of symbols across the corpus, weighting each word by its frequency.
3. Merge the most frequent pair into a new symbol, and record the merge.
4. Repeat until you have done the number of merges you want; vocabulary size = characters + merges.

To encode new text you replay the learned merges in the order they were learned. We implement this from scratch first, then train the fast Rust implementation from Hugging Face `tokenizers` for the real model.

In [ ]:
def merge_symbols(symbols, pair):
    # Replace every occurrence of the adjacent `pair` in `symbols` with the merged symbol.
    out, i = [], 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1]); i += 2
        else:
            out.append(symbols[i]); i += 1
    return tuple(out)

def learn_bpe(sentences, num_merges):
    # Byte-pair encoding on characters. Returns the merges in order and a history of
    # the average number of symbols per word (weighted by frequency) after each merge.
    word_freq = Counter(w for s in sentences for w in s.split())
    words = {tuple("▁" + w): f for w, f in word_freq.items()}   # each word as a tuple of symbols
    total = sum(word_freq.values())
    merges, history = [], []
    for _ in range(num_merges):
        pair_counts = Counter()
        for symbols, f in words.items():
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
        if not pair_counts:
            break
        best = max(pair_counts, key=pair_counts.get)       # the most frequent adjacent pair
        merges.append((best, pair_counts[best]))
        words = {merge_symbols(symbols, best): f for symbols, f in words.items()}
        history.append(sum(len(s) * f for s, f in words.items()) / total)
    return merges, history

def bpe_encode(sentence, merges):
    # Encode a sentence by replaying the merges, earliest (= most frequent) first.
    ranks = {pair: r for r, (pair, _) in enumerate(merges)}
    tokens = []
    for word in sentence.split():
        symbols = list("▁" + word)
        while len(symbols) > 1:
            rank, i = min((ranks.get(p, math.inf), i) for i, p in enumerate(zip(symbols, symbols[1:])))
            if rank == math.inf:          # no learned merge applies any more
                break
            symbols[i:i + 2] = [symbols[i] + symbols[i + 1]]
        tokens.extend(symbols)
    return tokens

def bpe_decode(tokens):
    return "".join(tokens).replace("▁", " ").strip()

In [ ]:
bpe_sample = [s for pair in raw["train"][:2000] for s in pair]     # 2,000 German + 2,000 English sentences
n_merges = 60 if QUICK else 300
t0 = time.time()
merges, history = learn_bpe(bpe_sample, n_merges)
print(f"learned {len(merges)} merges in {time.time() - t0:.1f}s")
print("first 20 merges (pair -> count):")
for (a, b), count in merges[:20]:
    print(f"  {a!r:>6} + {b!r:<6} -> {a + b!r:<9} {count:>6}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, len(history) + 1), history)
ax.set_xlabel("number of merges"); ax.set_ylabel("symbols per word")
ax.set_title("Each merge shortens the corpus: BPE is compression")
plt.show()

examples = ["Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.",
            "A little girl climbing into a wooden playhouse."]
for k in [10, 50, len(merges)]:
    print(f"--- after {k} merges")
    for s in examples:
        toks = bpe_encode(s, merges[:k])
        assert bpe_decode(toks) == s              # BPE is lossless: decoding gives the sentence back
        print(f"{len(toks):>3} tokens:", " ".join(toks))

Frequent words become single tokens quickly. Rare words such as `Antriebsradsystem` stay split into pieces the model has seen before, so nothing is ever out of vocabulary.

**The production tokenizer.** Our from-scratch version takes seconds per hundred merges. Hugging Face `tokenizers` runs the same algorithm in Rust and learns 8,000 merges in about a second. The settings:

- **NFKC normalisation**: canonical Unicode forms, so `ä` written as one or two code points becomes the same symbol.
- **Metaspace pre-tokenizer**: replaces each space with `▁` and splits there, like our version. Because the spaces are kept as `▁`, decoding is exact.
- **Punctuation split**: `Wiese.` becomes `▁Wiese` + `.`, so the model does not need a separate token for every word followed by a full stop.
- **Special tokens**, with fixed ids: `<pad>` = 0 (filler for equal-length batches), `<s>` = 1 (begin sentence), `</s>` = 2 (end sentence), `<unk>` = 3 (never used in practice, since every character in the training data is in the vocabulary).
- **One vocabulary shared by both languages** (8,000 tokens; the paper used 37,000 for 4.5M sentence pairs). We train it on the training split only, so the validation and test sentences stay unseen.

In [ ]:
SPECIALS = ["<pad>", "<s>", "</s>", "<unk>"]
PAD, BOS, EOS, UNK = 0, 1, 2, 3
VOCAB_SIZE = 2000 if QUICK else 8000

def train_tokenizer(texts, vocab_size):
    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.normalizer = normalizers.NFKC()
    tok.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Metaspace(replacement="▁", prepend_scheme="always"),
        pre_tokenizers.Punctuation(behavior="isolated"),
    ])
    tok.decoder = decoders.Metaspace(replacement="▁", prepend_scheme="always")
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,
                                  special_tokens=SPECIALS, show_progress=False)
    tok.train_from_iterator(texts, trainer=trainer)
    return tok

t0 = time.time()
tok = train_tokenizer([s for pair in raw["train"] for s in pair], VOCAB_SIZE)
assert [tok.token_to_id(s) for s in SPECIALS] == [PAD, BOS, EOS, UNK]
V = tok.get_vocab_size()
tok.save(str(DATA_DIR / f"bpe_shared_{V}.json"))
print(f"trained a shared BPE vocabulary of {V:,} tokens in {time.time() - t0:.1f}s")

In [ ]:
for s in [raw["train"][0][0], raw["train"][0][1], "Ein Hund rennt über die grüne Wiese.", "Antriebsradsystem"]:
    enc = tok.encode(s)
    print(f"{s}\n   tokens: {enc.tokens}\n   ids:    {enc.ids}")
    assert tok.decode(enc.ids) == s

# Round trip on every validation sentence, and how long sentences become in tokens
mismatched = [s for pair in raw["val"] for s in pair if tok.decode(tok.encode(s).ids) != s]
mismatches = len(mismatched)
src_tok = [len(tok.encode(s).ids) for s, _ in raw["train"]]
tgt_tok = [len(tok.encode(t).ids) for _, t in raw["train"]]
unk = sum(i == UNK for pair in raw["val"] for s in pair for i in tok.encode(s).ids)
print(f"\nvalidation round-trip mismatches: {mismatches} | <unk> tokens in validation: {unk}")
for s in mismatched[:3]:
    # NFKC normalisation is the usual cause: e.g. a non-breaking space becomes a plain space
    changed = sorted({f"{c!r} (U+{ord(c):04X})" for c in s if unicodedata.normalize("NFKC", c) != c})
    print(f"   changed by normalisation: {', '.join(changed) or 'whitespace'} in: {s[:70]}...")
print(f"tokens per sentence: {SRC_LANG} mean {np.mean(src_tok):.1f} (max {max(src_tok)}), "
      f"{TGT_LANG} mean {np.mean(tgt_tok):.1f} (max {max(tgt_tok)})")

In [ ]:
# How much of the shared vocabulary does each language use?
used_src = Counter(i for s, _ in raw["train"] for i in tok.encode(s).ids)
used_tgt = Counter(i for _, t in raw["train"] for i in tok.encode(t).ids)
both = set(used_src) & set(used_tgt)
print(f"tokens used by {SRC_LANG} only: {len(set(used_src) - both):,} | {TGT_LANG} only: {len(set(used_tgt) - both):,} | both: {len(both):,}")
word_start = lambda t: t.startswith("▁") and t[1:].isalpha() and len(t) >= 4   # a word-initial token, 3+ letters
shared_words = [tok.id_to_token(i) for i, _ in (used_src & used_tgt).most_common(600) if word_start(tok.id_to_token(i))][:15]
print("frequent word tokens used by both languages:", shared_words)
print("(some are true shared words such as names; others are false friends like German 'hat' = 'has')")

**Check yourself**

- Why does BPE never produce an out-of-vocabulary token for text made of characters it has seen?
- What happens to sequence length, and therefore to attention cost, as you shrink the vocabulary?
- Why must the tokenizer be trained on the training split only?

## Part 3 · Batches, teacher forcing and masks

**Ids with sentence markers.** Each source sentence becomes `ids + </s>`. Each target sentence becomes `<s> + ids + </s>`.

**Teacher forcing.** The decoder is trained to predict the next token given the correct previous tokens, not its own guesses. From one target sequence we make two views, shifted by one position:

```
tgt     = <s>  A    dog  runs  </s>
tgt_in  = <s>  A    dog  runs          (what the decoder reads)
tgt_out =      A    dog  runs  </s>    (what it must predict at each position)
```

All positions are trained in parallel in a single forward pass, which is the big speed advantage over an RNN. That only works if position *t* cannot see positions after *t*; the **causal mask** enforces that.

**Batching.** Sentences have different lengths, so a batch is padded with `<pad>` to its longest member. The paper groups sentence pairs of similar length and fills each batch with about 25,000 source and 25,000 target tokens. We do the same at a smaller scale: sort sentences by length within large random pools, then cut batches so that `sentences × longest length ≤ max_tokens`. Fewer pad tokens means less wasted compute.

In [ ]:
MAX_LEN = 100   # token cap per sentence; Multi30k sentences are far shorter, this only guards outliers

def encode_pairs(pairs):
    data = []
    for s, t in pairs:
        src = tok.encode(s).ids + [EOS]
        tgt = [BOS] + tok.encode(t).ids + [EOS]
        if len(src) <= MAX_LEN and len(tgt) <= MAX_LEN:
            data.append((src, tgt))
    return data

encoded = {split: encode_pairs(pairs) for split, pairs in raw.items()}
for split, data in encoded.items():
    print(f"{split:>5}: {len(data):,} pairs | longest source {max(len(s) for s, _ in data)} tokens, "
          f"longest target {max(len(t) for _, t in data)} tokens")

**Masks.** Attention scores have shape `(B, h, T_query, T_key)`. A mask says which (query, key) pairs are allowed; `True` means "may attend". We build masks that broadcast to that shape:

- **source mask** `(B, 1, 1, S)`: hides `<pad>` keys in the source. Used by encoder self-attention and by the decoder's attention over the encoder output.
- **target mask** `(B, 1, T, T)`: the AND of a pad mask on the target keys and the **causal (subsequent) mask**, a lower-triangular matrix that lets position *t* see only positions 0…*t*.

In [ ]:
def subsequent_mask(size, device=None):
    # (size, size) lower-triangular boolean matrix: query position t may attend to key positions <= t.
    return torch.tril(torch.ones(size, size, dtype=torch.bool, device=device))

class Batch:
    # One training batch: source, shifted target views and both masks.
    def __init__(self, src, tgt=None, pad=PAD):
        self.src = src                                           # (B, S)
        self.src_mask = (src != pad)[:, None, None, :]           # (B, 1, 1, S)
        if tgt is not None:
            self.tgt = tgt                                       # (B, T + 1), with <s> and </s>
            self.tgt_in = tgt[:, :-1]                            # (B, T) decoder input
            self.tgt_out = tgt[:, 1:]                            # (B, T) next-token targets
            self.tgt_mask = self.make_tgt_mask(self.tgt_in, pad) # (B, 1, T, T)
            self.ntokens = int((self.tgt_out != pad).sum())      # real (non-pad) target tokens

    @staticmethod
    def make_tgt_mask(tgt, pad):
        pad_mask = (tgt != pad)[:, None, None, :]                # (B, 1, 1, T): hide padded keys
        causal = subsequent_mask(tgt.size(-1), tgt.device)       # (T, T): hide future keys
        return pad_mask & causal[None, None]

    def to(self, device):
        for name in ("src", "src_mask", "tgt", "tgt_in", "tgt_out", "tgt_mask"):
            if hasattr(self, name):
                setattr(self, name, getattr(self, name).to(device, non_blocking=True))
        return self

print(subsequent_mask(5).int())

In [ ]:
def make_batches(data, max_tokens, shuffle=True, seed=0, pool_size=4096):
    # Token-based batching: indices grouped so that sentences x longest length <= max_tokens.
    rng = random.Random(seed)
    order = list(range(len(data)))
    if shuffle:
        rng.shuffle(order)
    batches = []
    pair_len = lambda i: max(len(data[i][0]), len(data[i][1]))     # what the token budget counts
    for start in range(0, len(order), pool_size):
        # sort by the same length the budget uses, so neighbours in a batch fit each other on both sides
        pool = sorted(order[start:start + pool_size], key=lambda i: (pair_len(i), len(data[i][0]) + len(data[i][1])))
        batch, longest = [], 0
        for i in pool:
            length = pair_len(i)
            if batch and (len(batch) + 1) * max(longest, length) > max_tokens:
                batches.append(batch); batch, longest = [], 0
            batch.append(i); longest = max(longest, length)
        if batch:
            batches.append(batch)
    if shuffle:
        rng.shuffle(batches)
    return batches

def collate(data, indices, pad=PAD):
    src = pad_sequence([torch.tensor(data[i][0]) for i in indices], batch_first=True, padding_value=pad)
    tgt = pad_sequence([torch.tensor(data[i][1]) for i in indices], batch_first=True, padding_value=pad)
    return Batch(src, tgt, pad)

def pad_fraction(data, batches, side):
    # Fraction of <pad> tokens on one side (0 = source, 1 = target) of the padded batches
    real = padded = 0
    for b in batches:
        lengths = [len(data[i][side]) for i in b]
        real += sum(lengths); padded += len(b) * max(lengths)
    return 1 - real / padded

train_batches = make_batches(encoded["train"], max_tokens=2048)
sizes = [len(b) for b in train_batches]
avg = int(np.mean(sizes))
rng = random.Random(0)
shuffled = list(range(len(encoded["train"]))); rng.shuffle(shuffled)
random_batches = [shuffled[i:i + avg] for i in range(0, len(shuffled), avg)]
print(f"{len(train_batches)} batches of ~{avg} sentences (min {min(sizes)}, max {max(sizes)})")
for name, bs in (("length bucketing", train_batches), ("random batches of the same size", random_batches)):
    print(f"padding with {name:<32} source {pad_fraction(encoded['train'], bs, 0):5.1%} | "
          f"target {pad_fraction(encoded['train'], bs, 1):5.1%}")

In [ ]:
batch = collate(encoded["train"], train_batches[0][:4])     # a small batch of 4 sentences, to read
for name in ("src", "src_mask", "tgt_in", "tgt_out", "tgt_mask"):
    describe(f"batch.{name}", getattr(batch, name))
print("ntokens (non-pad targets):", batch.ntokens)

print("\nsentence 0, teacher forcing view:")
for t_in, t_out in zip(batch.tgt_in[0].tolist(), batch.tgt_out[0].tolist()):
    print(f"   reads {tok.id_to_token(t_in):<14} -> must predict {tok.id_to_token(t_out)}")

In [ ]:
# Visualise the masks for two sentences of different lengths: white = blocked, blue = allowed
first = range(200)
i_long = max(first, key=lambda i: len(encoded["train"][i][1]))
i_short = min(first, key=lambda i: len(encoded["train"][i][1]))
b = collate(encoded["train"], [i_long, i_short])        # sentence 1 gets padded to sentence 0's length
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1, 1.3]})
axes[0].imshow(subsequent_mask(10).int(), cmap=SEQ_CMAP, vmin=0, vmax=1.4)
axes[0].set_title("causal mask (T = 10)")
axes[1].imshow(b.tgt_mask[1, 0].int(), cmap=SEQ_CMAP, vmin=0, vmax=1.4)
axes[1].set_title("target mask, shorter sentence\n(causal AND not-pad)")
axes[2].imshow(b.src_mask[:, 0, 0].int(), cmap=SEQ_CMAP, vmin=0, vmax=1.4, aspect="auto")
axes[2].set_title("source masks (rows = sentences)")
for ax in axes:
    ax.grid(False); ax.set_xlabel("key position")
axes[0].set_ylabel("query position"); axes[1].set_ylabel("query position")
axes[2].set_yticks([0, 1]); axes[2].set_yticklabels(["sentence 0", "sentence 1"])
plt.tight_layout(); plt.show()

**Reading the middle panel.** The staircase at the top is the causal mask over the shorter sentence's real tokens. The rows below it are its `<pad>` positions, and they're allowed to see the real tokens on purpose: a row that blocks every key has nothing to attend to (with a −∞ fill its softmax is 0/0 = `NaN`, and `NaN` spreads through everything after it). Those rows still produce outputs, but the loss ignores `<pad>` targets, so they never affect training.

## Part 4 · The model, piece by piece

The Transformer is an **encoder–decoder**. The encoder reads the whole source sentence and produces one vector per source token, the **memory**. The decoder writes the target one token at a time; at every step it looks at the tokens it has written so far and, through attention, at the memory.

```
 source ids (B, S)                                   target ids so far (B, T)
       │                                                       │
 Embedding × √d_model + positional encoding          Embedding × √d_model + positional encoding
       │                                                       │
 ┌─────▼──────────────────────┐                      ┌─────────▼──────────────────────────┐
 │ Encoder layer  (× N)       │                      │ Decoder layer  (× N)               │
 │  self-attention            │                      │  masked self-attention (causal)    │
 │  add & norm                │   memory (B, S, d)   │  add & norm                        │
 │  feed-forward              │ ───────────────────► │  attention over memory (cross)     │
 │  add & norm                │                      │  add & norm                        │
 └────────────────────────────┘                      │  feed-forward                      │
                                                     │  add & norm                        │
                                                     └─────────┬──────────────────────────┘
                                                               │ (B, T, d_model)
                                                     Linear (tied to the embeddings) + softmax
                                                               │
                                                     next-token probabilities (B, T, V)
```

**From your ViT background:** an encoder layer is exactly a ViT block (self-attention + MLP, each wrapped in a residual and a LayerNorm), with token embeddings instead of patch embeddings and no `[CLS]` token. The decoder layer is the same block with two changes: its self-attention is **causal**, and it has an extra **cross-attention** whose queries come from the decoder and whose keys and values come from the encoder's memory.

We build every box in that diagram, test each one, then assemble them.

### 4.1 Embeddings

An embedding is a lookup table: row *i* of a `(V, d_model)` matrix is the vector for token *i*. The paper multiplies embeddings by $\sqrt{d_\text{model}}$ (section 3.4). We initialise the table with standard deviation $d_\text{model}^{-1/2}$, so after the scaling each component has a standard deviation of about 1, the same scale as the positional encodings added next. Without the scaling, the positional signal would drown out the token identity.

In [ ]:
class Embeddings(nn.Module):
    # Token ids -> vectors, multiplied by sqrt(d_model) (section 3.4).
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.lut = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):                                   # x: (B, L) token ids
        return self.lut(x) * math.sqrt(self.d_model)        # (B, L, d_model)

emb = Embeddings(V, 512)
nn.init.normal_(emb.lut.weight, std=512 ** -0.5)
x_emb = emb(batch.src)
describe("source ids", batch.src)
describe("embedded source", x_emb)
print(f"std of embedded values: {x_emb.std():.3f}  (about 1 by design)")

### 4.2 Positional encoding

Attention by itself is **permutation-equivariant**: shuffle the input tokens and the outputs shuffle the same way. Nothing in it knows about order. Order has to be added to the input, as a position-dependent vector added to each embedding. The paper uses fixed sinusoids:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_\text{model}}}\right), \qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_\text{model}}}\right)$$

Each pair of dimensions $(2i, 2i+1)$ is a point rotating on a circle at its own frequency $\omega_i = 10000^{-2i/d_\text{model}}$. The wavelengths form a geometric progression from $2\pi$ up to $10000 \cdot 2\pi$, so the first dimensions change fast and the last ones barely move. Together they work like the hands of a clock: every position gets a distinct pattern.

The paper's reason for this choice: for any fixed offset $k$, $PE_{pos+k}$ is a **linear function** of $PE_{pos}$ (a rotation of each pair by $k\,\omega_i$), which should make it easy to attend by relative position. We check that numerically below. Dropout (rate 0.1) is applied to the sum of embedding and encoding.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    # Fixed sine/cosine positional encodings (section 3.5), added to the embeddings.
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)          # (max_len, 1)
        inv_freq = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32)
                             * (-math.log(10000.0) / d_model))                       # (d_model/2,) = omega_i
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * inv_freq)                                 # even dims: sine
        pe[:, 1::2] = torch.cos(position * inv_freq)                                 # odd dims: cosine
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)                # (1, max_len, d_model)

    def forward(self, x):                                                            # (B, L, d_model)
        return self.dropout(x + self.pe[:, : x.size(1)])

class LearnedPositionalEncoding(nn.Module):
    # The alternative the paper tested: one trained vector per position (Table 3, row E).
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.pos = nn.Embedding(max_len, d_model)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return self.dropout(x + self.pos(positions)[None])

pe = SinusoidalPositionalEncoding(64, dropout=0.0).pe[0]        # (max_len, 64)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1.3, 1]})
im = axes[0].imshow(pe[:100].T, cmap=DIV_CMAP, vmin=-1, vmax=1, aspect="auto")
axes[0].set_xlabel("position"); axes[0].set_ylabel("dimension"); axes[0].grid(False)
axes[0].set_title("PE for 100 positions (d_model = 64)")
fig.colorbar(im, ax=axes[0], fraction=0.03)
for d in (4, 5, 6, 7):
    axes[1].plot(pe[:100, d], label=f"dim {d}")
axes[1].set_xlabel("position"); axes[1].set_title("Dimensions 4–7: sine/cosine pairs")
axes[1].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Check the paper's claim: PE[pos + k] = PE[pos] @ R_k, where R_k rotates each (sin, cos) pair by k * omega_i.
d = 64
inv_freq = torch.exp(torch.arange(0, d, 2, dtype=torch.float64) * (-math.log(10000.0) / d))
pos = torch.arange(200, dtype=torch.float64)[:, None]
pe64 = torch.zeros(200, d, dtype=torch.float64)
pe64[:, 0::2], pe64[:, 1::2] = torch.sin(pos * inv_freq), torch.cos(pos * inv_freq)

def shift_matrix(k):
    R = torch.zeros(d, d, dtype=torch.float64)
    for i, w in enumerate(inv_freq.tolist()):
        c, s = math.cos(k * w), math.sin(k * w)
        R[2 * i:2 * i + 2, 2 * i:2 * i + 2] = torch.tensor([[c, -s], [s, c]], dtype=torch.float64)
    return R

for k in (1, 7, 50):
    err = (pe64[:100] @ shift_matrix(k) - pe64[k:100 + k]).abs().max().item()
    print(f"offset k = {k:>2}: max |PE[pos] R_k - PE[pos + k]| = {err:.1e}")

# The dot product PE[i] . PE[j] depends only on the distance |i - j|: sum_i cos((i - j) * omega_i)
sim = pe64[:60] @ pe64[:60].T
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].imshow(sim, cmap=SEQ_CMAP); axes[0].grid(False)
axes[0].set_title("PE[i] · PE[j] for 60 positions"); axes[0].set_xlabel("j"); axes[0].set_ylabel("i")
axes[1].plot(sim[20].numpy())
axes[1].set_title("row i = 20: peaks at j = 20, falls off with distance"); axes[1].set_xlabel("j")
plt.tight_layout(); plt.show()

### 4.3 Scaled dot-product attention

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Read it as a **soft dictionary lookup**. Every token produces three vectors:

- a **query** $q$: what this token is looking for
- a **key** $k$: what this token offers, used to decide whether it is a match
- a **value** $v$: what this token hands over if it is chosen

The score of query $i$ against key $j$ is the dot product $q_i \cdot k_j$: large when they point the same way. A softmax over the keys turns each query's scores into weights that sum to 1, and the output for query $i$ is the weighted average of the values. A hard dictionary returns the value of the one key that matches exactly; attention returns a blend of all values, weighted by how well each key matches, and the blend is differentiable, so the model can learn what to look for.

**Masking.** Blocked (query, key) pairs get a score of $-\infty$ before the softmax, so their weight is exactly 0. We use the most negative finite number instead of $-\infty$: if a row were ever fully blocked, it then gives uniform weights rather than `NaN`.

In [ ]:
def attention(query, key, value, mask=None, dropout=None, scale=True):
    # Scaled dot-product attention, equation (1).
    # query (..., T_q, d_k), key (..., T_k, d_k), value (..., T_k, d_v); mask: True = may attend.
    d_k = query.size(-1)
    scores = query @ key.transpose(-2, -1)                    # (..., T_q, T_k): every query against every key
    if scale:
        scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
    p_attn = scores.softmax(dim=-1)                           # each row sums to 1 over the keys
    if dropout is not None:
        p_attn = dropout(p_attn)
    return p_attn @ value, p_attn                             # (..., T_q, d_v), (..., T_q, T_k)

In [ ]:
# A worked example: 3 tokens, d_model = 4, projected to d_k = d_v = 2. Self-attention: Q, K, V all come from X.
torch.manual_seed(0)
X = torch.randn(3, 4)                                         # 3 token vectors
W_q, W_k, W_v = torch.randn(4, 2), torch.randn(4, 2), torch.randn(4, 2)
Q, K, Vm = X @ W_q, X @ W_k, X @ W_v
torch.set_printoptions(precision=3, sci_mode=False)
print("Q =\n", Q, "\nK =\n", K, "\nV =\n", Vm)
scores = Q @ K.T / math.sqrt(2)
print("\nscores = Q K^T / sqrt(d_k): row i = query i, column j = key j\n", scores)
weights = scores.softmax(-1)
print("\nsoftmax over each row (rows sum to 1)\n", weights)
out = weights @ Vm
print("\noutput = weights @ V: each row is a blend of the value rows\n", out)
out2, w2 = attention(Q, K, Vm)
assert torch.allclose(out, out2) and torch.allclose(w2.sum(-1), torch.ones(3))

In [ ]:
# The soft-dictionary picture with hand-made keys: three "addresses", each storing a value.
keys = torch.tensor([[4.0, 0.0, 0.0], [0.0, 4.0, 0.0], [0.0, 0.0, 4.0]])
values = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
for q in ([0.0, 1.0, 0.0], [0.0, 1.0, 1.0], [0.0, 0.0, 0.0]):
    out, w = attention(torch.tensor([q]), keys, values, scale=False)
    print(f"query {q}: weights {w[0].numpy().round(3)} -> output {out[0].numpy().round(3)}")
print("-> a clear match returns that key's value; a tie blends; a query matching nothing averages everything")

**Why divide by $\sqrt{d_k}$?** If the components of $q$ and $k$ are independent with mean 0 and variance 1, then $q \cdot k$ has mean 0 and **variance $d_k$** (footnote 4 of the paper). With $d_k = 64$, raw scores have a standard deviation of 8; the softmax of numbers that far apart is nearly one-hot, and a nearly one-hot softmax has nearly zero gradient (its Jacobian $\mathrm{diag}(p) - pp^\top$ vanishes). Dividing by $\sqrt{d_k}$ brings the variance back to 1. The experiment below measures exactly this.

In [ ]:
rows = []
for d_k in (4, 16, 64, 256, 1024):
    q, k = torch.randn(2000, 1, d_k), torch.randn(2000, 32, d_k)      # 2,000 queries, 32 keys each
    raw_scores = (q * k).sum(-1)                                       # (2000, 32)
    for scaled in (False, True):
        p = (raw_scores / math.sqrt(d_k) if scaled else raw_scores).softmax(-1)
        jac = torch.diag_embed(p) - p.unsqueeze(-1) * p.unsqueeze(-2)  # softmax Jacobian per query
        rows.append(dict(d_k=d_k, scaled=scaled, score_var=raw_scores.var().item(),
                         max_weight=p.max(-1).values.mean().item(),
                         grad_norm=jac.flatten(1).norm(dim=-1).mean().item()))
print(f"{'d_k':>5} {'scaled':>7} {'var(q.k)':>9} {'mean max weight':>16} {'softmax grad norm':>18}")
for r in rows:
    print(f"{r['d_k']:>5} {str(r['scaled']):>7} {r['score_var']:>9.1f} {r['max_weight']:>16.3f} {r['grad_norm']:>18.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for scaled, label in ((False, "QK^T"), (True, "QK^T / sqrt(d_k)")):
    sel = [r for r in rows if r["scaled"] == scaled]
    axes[0].plot([r["d_k"] for r in sel], [r["max_weight"] for r in sel], marker="o", label=label)
    axes[1].plot([r["d_k"] for r in sel], [r["grad_norm"] for r in sel], marker="o", label=label)
for ax, title in zip(axes, ("mean largest attention weight", "softmax gradient size")):
    ax.set_xscale("log", base=2); ax.set_xlabel("d_k"); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Test: our attention equals PyTorch's fused kernel (which on the GPU can run FlashAttention).
torch.manual_seed(0)
q, k, v = torch.randn(2, 8, 5, 16), torch.randn(2, 8, 7, 16), torch.randn(2, 8, 7, 16)
mask = torch.rand(2, 1, 5, 7) > 0.3
mask[..., 0] = True                                           # every query keeps at least one key
ours, _ = attention(q, k, v, mask)
reference = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
print("max difference vs F.scaled_dot_product_attention:", (ours - reference).abs().max().item())
assert torch.allclose(ours, reference, atol=1e-5)

### 4.4 Multi-head attention

One attention computes one weighted average per query, so it can follow only one kind of relationship at a time. **Multi-head attention** runs $h$ attentions in parallel on different learned projections and concatenates the results:

$$\mathrm{MultiHead}(Q, K, V) = \mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_h)\,W^O, \qquad \mathrm{head}_i = \mathrm{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

With $d_k = d_v = d_\text{model}/h$, the total cost matches a single full-width head. The paper: *"Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this."*

In code, the $h$ projections $W_i^Q$ are one `Linear(d_model, d_model)` whose output is cut into $h$ slices of width $d_k$. The reshape `(B, L, d_model) → (B, L, h, d_k) → (B, h, L, d_k)` puts the heads in their own batch dimension so all of them run in one matrix multiply.

The same module is used three ways (section 3.2.3):

| Use | Queries from | Keys and values from | Mask |
|---|---|---|---|
| encoder self-attention | encoder layer input | the same | source pad mask |
| decoder self-attention | decoder layer input | the same | target mask (pad + causal) |
| encoder–decoder (cross) attention | decoder | encoder memory | source pad mask |

`use_sdpa=True` routes the inner attention through `F.scaled_dot_product_attention` (fused kernels, faster and lighter on the 4090). Our own `attention()` runs whenever we want to keep the weights for plotting.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h, dropout=0.1, scale=True, use_sdpa=True):
        super().__init__()
        assert d_model % h == 0, "d_model must be divisible by the number of heads"
        self.d_k, self.h = d_model // h, h
        self.w_q = nn.Linear(d_model, d_model)      # all h query projections W_i^Q, side by side
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)      # W^O
        self.dropout = nn.Dropout(dropout)          # dropout on the attention weights
        self.scale, self.use_sdpa = scale, use_sdpa
        self.store_attn, self.attn = False, None    # set store_attn = True to keep weights for plots

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        def split_heads(x):                          # (B, L, d_model) -> (B, h, L, d_k)
            return x.view(B, -1, self.h, self.d_k).transpose(1, 2)
        q, k, v = split_heads(self.w_q(query)), split_heads(self.w_k(key)), split_heads(self.w_v(value))
        if self.use_sdpa and not self.store_attn:
            x = F.scaled_dot_product_attention(
                q, k, v, attn_mask=mask, dropout_p=self.dropout.p if self.training else 0.0,
                scale=None if self.scale else 1.0)
        else:
            x, p_attn = attention(q, k, v, mask, self.dropout, self.scale)
            if self.store_attn:
                self.attn = p_attn.detach().float().cpu()          # (B, h, T_q, T_k)
        x = x.transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)   # concatenate heads
        return self.w_o(x)                                          # (B, T_q, d_model)

In [ ]:
# Trace one forward pass by hand: B = 2 sentences, L = 5 tokens, d_model = 16, h = 4 heads, d_k = 4
torch.manual_seed(0)
d_model, h = 16, 4
mha = MultiHeadAttention(d_model, h, dropout=0.0)
x = torch.randn(2, 5, d_model)
Bsz, d_k = x.size(0), d_model // h
describe("x", x)
q = mha.w_q(x);                                    describe("q = x W_q", q)
q_h = q.view(Bsz, -1, h, d_k).transpose(1, 2);     describe("q split into heads", q_h)
k_h = mha.w_k(x).view(Bsz, -1, h, d_k).transpose(1, 2)
v_h = mha.w_v(x).view(Bsz, -1, h, d_k).transpose(1, 2)
scores = q_h @ k_h.transpose(-2, -1) / math.sqrt(d_k); describe("scores per head", scores)
p = scores.softmax(-1);                            describe("attention weights", p)
ctx = p @ v_h;                                     describe("weighted values per head", ctx)
concat = ctx.transpose(1, 2).contiguous().view(Bsz, -1, h * d_k); describe("heads concatenated", concat)
out = mha.w_o(concat);                             describe("output = concat W_o", out)
assert torch.allclose(out, mha(x, x, x), atol=1e-6)
n_params = sum(p_.numel() for p_ in mha.parameters())
print(f"\nparameters: {n_params} = 4 x (d_model^2 + d_model) = {4 * (d_model**2 + d_model)}")

In [ ]:
# Test against PyTorch's nn.MultiheadAttention, with our weights copied in.
reference = nn.MultiheadAttention(d_model, h, dropout=0.0, batch_first=True)
with torch.no_grad():
    reference.in_proj_weight.copy_(torch.cat([mha.w_q.weight, mha.w_k.weight, mha.w_v.weight]))
    reference.in_proj_bias.copy_(torch.cat([mha.w_q.bias, mha.w_k.bias, mha.w_v.bias]))
    reference.out_proj.weight.copy_(mha.w_o.weight)
    reference.out_proj.bias.copy_(mha.w_o.bias)
keep = torch.ones(2, 5, dtype=torch.bool)
keep[1, 3:] = False                                    # sentence 1 ends with 2 pad tokens
ours = mha(x, x, x, keep[:, None, None, :])
theirs, _ = reference(x, x, x, key_padding_mask=~keep) # PyTorch's convention: True = ignore
print("max difference vs nn.MultiheadAttention:", (ours - theirs).abs().max().item())
assert torch.allclose(ours, theirs, atol=1e-5)

# Our explicit attention path and the fused SDPA path agree too
mha.use_sdpa = False
explicit = mha(x, x, x, keep[:, None, None, :])
mha.use_sdpa = True
assert torch.allclose(explicit, ours, atol=1e-5)
print("explicit path == fused path")

### 4.5 Position-wise feed-forward network

$$\mathrm{FFN}(x) = \max(0,\; xW_1 + b_1)\,W_2 + b_2$$

The same two-layer MLP is applied to every position independently (equivalently, two convolutions with kernel size 1). The inner width is $d_\text{ff} = 2048 = 4 \times d_\text{model}$. Attention mixes information **across** positions; the FFN transforms each position's vector **on its own**, and holds about two thirds of each encoder layer's parameters.

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):                                     # (B, L, d_model)
        return self.w_2(self.dropout(F.relu(self.w_1(x))))   # (B, L, d_ff) inside, (B, L, d_model) out

ff = PositionwiseFeedForward(512, 2048, dropout=0.0)
xf = torch.randn(2, 7, 512)
describe("FFN output", ff(xf))
# "Position-wise": position 3's output depends only on position 3's input
assert torch.allclose(ff(xf)[:, 3], ff(xf[:, 3:4])[:, 0], atol=1e-6)
print(f"parameters: {sum(p.numel() for p in ff.parameters()):,} vs 4 * 512^2 + 512 * 4 for attention: {4 * 512**2 + 4 * 512:,}")

### 4.6 Residual connections, dropout and LayerNorm

Every sub-layer (attention or FFN) is wrapped as

$$\text{output} = \mathrm{LayerNorm}\big(x + \mathrm{Dropout}(\mathrm{Sublayer}(x))\big)$$

- The **residual** $x + \ldots$ gives gradients a direct path through the stack: each layer learns a correction to its input rather than a whole new representation.
- **Dropout** ($P_\text{drop} = 0.1$) is applied to each sub-layer's output before the residual add, and to the sum of embeddings and positional encodings (section 5.4).
- **LayerNorm** normalises each position's vector to mean 0 and variance 1 across its $d_\text{model}$ features, then applies a learned scale $\gamma$ and shift $\beta$. Unlike BatchNorm, it doesn't depend on the other sentences in the batch, so it behaves the same in training and at inference.

That arrangement is **post-norm** (normalise after the residual add), as in the paper. Most later models, and The Annotated Transformer's code, use **pre-norm**: $x + \mathrm{Dropout}(\mathrm{Sublayer}(\mathrm{LayerNorm}(x)))$, with one final LayerNorm after the last layer. Pre-norm keeps a clean identity path and trains deep stacks more stably with less warmup; post-norm is the original. Our model supports both (`norm_first`), and Part 12 compares them.

In [ ]:
class LayerNormScratch(nn.Module):
    # LayerNorm from scratch: normalise over the last dimension, then scale and shift.
    def __init__(self, features, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta

xl = torch.randn(3, 7, 16) * 5 + 2
assert torch.allclose(LayerNormScratch(16)(xl), nn.LayerNorm(16)(xl), atol=1e-5)
normed = LayerNormScratch(16)(xl)
print(f"after LayerNorm: per-position mean {normed.mean(-1).abs().max():.1e}, std {normed.std(-1, unbiased=False).mean():.3f}")
print("matches nn.LayerNorm; the model uses nn.LayerNorm (fused kernel)")

class SublayerConnection(nn.Module):
    # Residual + dropout + LayerNorm around any sub-layer. Post-norm (paper) or pre-norm.
    def __init__(self, d_model, dropout, norm_first=False):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.norm_first = norm_first

    def forward(self, x, sublayer):
        if self.norm_first:
            return x + self.dropout(sublayer(self.norm(x)))      # pre-norm
        return self.norm(x + self.dropout(sublayer(x)))          # post-norm, as in the paper

def clones(module, n):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(n)])

### 4.7 The encoder

An encoder layer is self-attention followed by the FFN, each wrapped in a `SublayerConnection`. The encoder is $N = 6$ identical layers (identical in shape; each has its own weights). With pre-norm, a final LayerNorm closes the stack; with post-norm, each layer's output is already normalised.

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, self_attn, feed_forward, dropout, norm_first=False):
        super().__init__()
        self.self_attn, self.feed_forward, self.size = self_attn, feed_forward, d_model
        self.sublayer = clones(SublayerConnection(d_model, dropout, norm_first), 2)

    def forward(self, x, mask):                                               # x: (B, S, d_model)
        x = self.sublayer[0](x, lambda y: self.self_attn(y, y, y, mask))      # tokens look at each other
        return self.sublayer[1](x, self.feed_forward)                         # each token transformed alone

class Encoder(nn.Module):
    def __init__(self, layer, n, norm_first=False):
        super().__init__()
        self.layers = clones(layer, n)
        self.norm = nn.LayerNorm(layer.size) if norm_first else nn.Identity()

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)                                                   # the memory: (B, S, d_model)

### 4.8 The decoder

A decoder layer has three sub-layers:

1. **masked self-attention** over the target tokens so far, with the causal mask, so position $t$ cannot peek at $t+1, t+2, \dots$
2. **encoder–decoder attention**: queries from the decoder, keys and values from the encoder memory. This is where the translation "looks at" the source sentence.
3. the **feed-forward** network

The masking, together with the one-position shift between `tgt_in` and `tgt_out`, guarantees that the prediction for position $t$ depends only on the outputs at positions before $t$.

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, self_attn, src_attn, feed_forward, dropout, norm_first=False):
        super().__init__()
        self.self_attn, self.src_attn, self.feed_forward, self.size = self_attn, src_attn, feed_forward, d_model
        self.sublayer = clones(SublayerConnection(d_model, dropout, norm_first), 3)

    def forward(self, x, memory, src_mask, tgt_mask):                         # x: (B, T, d_model)
        x = self.sublayer[0](x, lambda y: self.self_attn(y, y, y, tgt_mask))  # causal self-attention
        x = self.sublayer[1](x, lambda y: self.src_attn(y, memory, memory, src_mask))  # look at the source
        return self.sublayer[2](x, self.feed_forward)

class Decoder(nn.Module):
    def __init__(self, layer, n, norm_first=False):
        super().__init__()
        self.layers = clones(layer, n)
        self.norm = nn.LayerNorm(layer.size) if norm_first else nn.Identity()

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)                                                   # (B, T, d_model)

### 4.9 The generator and the full model

The **generator** maps each decoder output vector to a distribution over the vocabulary: a linear layer to $V$ logits, then (log-)softmax.

**Weight tying** (section 3.4): *"we share the same weight matrix between the two embedding layers and the pre-softmax linear transformation."* With a shared vocabulary all three are $V \times d_\text{model}$ matrices, so one matrix serves the source embedding, the target embedding and the output projection. The output logit for token $w$ is then the dot product between the decoder state and $w$'s own embedding. It saves $2 V d_\text{model}$ parameters (about 8M here) and usually helps on small data.

**Initialisation.** Matrices get Xavier-uniform initialisation (as in The Annotated Transformer); the embedding table gets $\mathcal{N}(0, d_\text{model}^{-1})$, matching the $\sqrt{d_\text{model}}$ scaling from 4.1.

One addition the paper doesn't have: the last projection of every residual branch (attention's $W^O$ and the FFN's second layer) is scaled by $1/\sqrt{2N}$ at initialisation, as in GPT-2. Without it, a 6-layer post-norm stack starts out nearly blind to position. Each sub-layer adds something that is the same at every position (near-uniform attention averages the values; ReLU outputs have a positive mean), and after 12 sub-layers the encoder's outputs at different positions of a sentence are about 96% alike. The decoder's cross-attention then starts out uniform, and with our small batches training never recovered: the first full run of this notebook stalled at 15 BLEU with uniform cross-attention. Scaled branches start each layer close to the identity, so positions stay distinct. The cell after the parameter count measures both.

The paper's two configurations (Table 3):

| | N | d_model | d_ff | h | d_k | P_drop | ε_ls | train steps |
|---|---|---|---|---|---|---|---|---|
| base | 6 | 512 | 2048 | 8 | 64 | 0.1 | 0.1 | 100K (12 h on 8 P100s) |
| big | 6 | 1024 | 4096 | 16 | 64 | 0.3 | 0.1 | 300K (3.5 days) |

In [ ]:
class Generator(nn.Module):
    # Decoder vectors -> log-probabilities over the vocabulary.
    def __init__(self, d_model, vocab_size, bias=True):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size, bias=bias)

    def forward(self, x):                                     # (B, T, d_model)
        return F.log_softmax(self.proj(x), dim=-1)            # (B, T, V)

class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder
        self.src_embed, self.tgt_embed, self.generator = src_embed, tgt_embed, generator

    def encode(self, src, src_mask):                          # (B, S) -> memory (B, S, d_model)
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):        # (B, T) -> (B, T, d_model)
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask, tgt_mask):          # returns decoder vectors; call .generator on them
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

@dataclass
class ModelConfig:
    src_vocab: int
    tgt_vocab: int
    N: int = 6
    d_model: int = 512
    d_ff: int = 2048
    h: int = 8
    dropout: float = 0.1
    norm_first: bool = False          # False = post-norm (paper); True = pre-norm
    pos_encoding: str = "sinusoidal"  # or "learned"
    tie_embeddings: bool = True       # share source/target embeddings and the output projection
    scale_attention: bool = True      # the 1/sqrt(d_k) in equation (1)
    use_sdpa: bool = True             # fused attention kernel when we don't need the weights
    scale_residual_init: bool = True  # residual branches' last projection x 1/sqrt(2N) at init (not in the paper)
    max_len: int = 512

def make_model(cfg):
    c = copy.deepcopy
    attn = MultiHeadAttention(cfg.d_model, cfg.h, cfg.dropout, cfg.scale_attention, cfg.use_sdpa)
    ff = PositionwiseFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
    PE = SinusoidalPositionalEncoding if cfg.pos_encoding == "sinusoidal" else LearnedPositionalEncoding
    position = PE(cfg.d_model, cfg.dropout, cfg.max_len)
    model = EncoderDecoder(
        Encoder(EncoderLayer(cfg.d_model, c(attn), c(ff), cfg.dropout, cfg.norm_first), cfg.N, cfg.norm_first),
        Decoder(DecoderLayer(cfg.d_model, c(attn), c(attn), c(ff), cfg.dropout, cfg.norm_first), cfg.N, cfg.norm_first),
        nn.Sequential(Embeddings(cfg.src_vocab, cfg.d_model), c(position)),
        nn.Sequential(Embeddings(cfg.tgt_vocab, cfg.d_model), c(position)),
        Generator(cfg.d_model, cfg.tgt_vocab, bias=not cfg.tie_embeddings),
    )
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    if cfg.scale_residual_init:
        with torch.no_grad():
            for m in model.modules():
                if isinstance(m, MultiHeadAttention):
                    m.w_o.weight.mul_((2 * cfg.N) ** -0.5)
                elif isinstance(m, PositionwiseFeedForward):
                    m.w_2.weight.mul_((2 * cfg.N) ** -0.5)
    for table in (model.src_embed[0].lut, model.tgt_embed[0].lut):
        nn.init.normal_(table.weight, mean=0.0, std=cfg.d_model ** -0.5)
    if cfg.tie_embeddings:
        assert cfg.src_vocab == cfg.tgt_vocab, "tying needs one shared vocabulary"
        model.src_embed[0].lut.weight = model.tgt_embed[0].lut.weight
        model.generator.proj.weight = model.tgt_embed[0].lut.weight
    model.cfg = cfg
    return model

def count_params(module):
    return sum(p.numel() for p in module.parameters())   # shared (tied) tensors are counted once

In [ ]:
# Parameter budget of the paper's base model: with its ~37k vocabulary, and with our 8k vocabulary
with torch.device("meta"):                     # build without allocating memory, just to count
    paper_base = make_model(ModelConfig(37000, 37000))
    our_base = make_model(ModelConfig(V, V))
for name, m in (("paper base, V = 37,000", paper_base), (f"our base, V = {V:,}", our_base)):
    print(f"{name:<24} total {count_params(m) / 1e6:5.1f}M | embeddings {count_params(m.tgt_embed) / 1e6:4.1f}M | "
          f"encoder {count_params(m.encoder) / 1e6:4.1f}M | decoder {count_params(m.decoder) / 1e6:4.1f}M")
print("(the paper reports 65M for base without saying exactly how it counts, so a small gap is expected)")
assert our_base.generator.proj.weight is our_base.tgt_embed[0].lut.weight is our_base.src_embed[0].lut.weight
print("weight tying: source embedding, target embedding and output projection are one tensor")

In [ ]:
# Why the scaled initialisation: at initialisation, how alike are the encoder's outputs at different
# positions of the same sentence? (cosine similarity; 1 = every position carries the same vector)
@torch.no_grad()
def position_similarity(model, batch):
    model.eval()
    memory = F.normalize(model.encode(batch.src, batch.src_mask), dim=-1)    # (B, S, d_model), unit vectors
    sims = []
    for b in range(memory.size(0)):
        m = memory[b, batch.src[b] != PAD]                                   # this sentence's real positions
        n = m.size(0)
        sims.append(((m @ m.T).sum() - n) / (n * (n - 1)))                   # mean over pairs i != j
    return torch.stack(sims).mean().item()

probe_batch = collate(encoded["val"], list(range(64)))
for scaled in (False, True):
    seed_everything(0)
    untrained = make_model(ModelConfig(V, V, scale_residual_init=scaled))
    print(f"scale_residual_init={scaled!s:<5} -> encoder outputs at different positions: cosine similarity "
          f"{position_similarity(untrained, probe_batch):.2f}")
del untrained

### 4.10 Tests on the assembled model

Four checks on a small randomly initialised model, in `eval()` mode so dropout is off:

1. **Shape trace**: forward hooks print every module's output shape in execution order. This is the quickest way to follow the data through the network.
2. **Causality**: changing target tokens after position $t$ must not change the decoder's outputs at positions $\le t$.
3. **Padding invariance**: appending `<pad>` tokens to the source must not change the encoder's outputs at the real positions.
4. **Fused attention = explicit attention** for the whole model.

In [ ]:
def trace_shapes(model, *inputs, max_depth=3):
    # Run one forward pass and print each module's output shape, in execution order.
    rows, hooks = [], []
    for name, module in model.named_modules():
        if name and name.count(".") < max_depth:
            hooks.append(module.register_forward_hook(
                lambda m, inp, out, name=name: rows.append((name, type(m).__name__, tuple(out.shape)))))
    try:
        with torch.no_grad():
            model(*inputs)
    finally:
        for hk in hooks:
            hk.remove()
    for name, kind, shape in rows:
        print(f"{name:<36} {kind:<30} {shape}")

seed_everything(0)
tiny = make_model(ModelConfig(V, V, N=2, d_model=64, d_ff=128, h=4)).eval()
b = collate(encoded["train"], train_batches[0][:3])
trace_shapes(tiny, b.src, b.tgt_in, b.src_mask, b.tgt_mask)

In [ ]:
with torch.no_grad():
    # 2. causality
    memory = tiny.encode(b.src, b.src_mask)
    out1 = tiny.decode(memory, b.src_mask, b.tgt_in, b.tgt_mask)
    t = 4
    changed = b.tgt_in.clone()
    changed[:, t + 1:] = torch.randint(4, V, changed[:, t + 1:].shape)
    out2 = tiny.decode(memory, b.src_mask, changed, Batch.make_tgt_mask(changed, PAD))
    assert torch.allclose(out1[:, :t + 1], out2[:, :t + 1], atol=1e-5)
    assert not torch.allclose(out1[:, t + 1:], out2[:, t + 1:], atol=1e-3)
    print(f"causality: outputs at positions <= {t} unchanged when later target tokens change")

    # 3. padding invariance
    src = b.src[:1]
    src_padded = torch.cat([src, torch.full((1, 5), PAD)], dim=1)
    m1 = tiny.encode(src, (src != PAD)[:, None, None, :])
    m2 = tiny.encode(src_padded, (src_padded != PAD)[:, None, None, :])
    assert torch.allclose(m1, m2[:, : src.size(1)], atol=1e-5)
    print("padding invariance: extra <pad> tokens leave the encoder outputs unchanged")

    # 4. fused SDPA path == explicit path for the whole model
    fused = tiny(b.src, b.tgt_in, b.src_mask, b.tgt_mask)
    for m in tiny.modules():
        if isinstance(m, MultiHeadAttention):
            m.use_sdpa = False
    explicit = tiny(b.src, b.tgt_in, b.src_mask, b.tgt_mask)
    for m in tiny.modules():
        if isinstance(m, MultiHeadAttention):
            m.use_sdpa = True
    assert torch.allclose(fused, explicit, atol=1e-5)
    print("fused attention == explicit attention for the full model")

    logprobs = tiny.generator(fused)
    describe("log-probabilities", logprobs)
    print(f"untrained model: mean probability of the correct next token = "
          f"{logprobs.exp().gather(-1, b.tgt_out[..., None]).mean():.5f} (uniform would be 1/V = {1 / V:.5f})")

## Part 5 · Training machinery

Training minimises the cross-entropy of the next token: at every target position, $-\log p(\text{correct token})$, averaged over the real (non-pad) tokens in the batch. Three details from section 5 of the paper shape how well that works:

1. **Label smoothing** (5.4) softens the targets.
2. **Adam with a warmup-then-decay learning rate** (5.3).
3. **Large, length-bucketed batches** (5.1), built in Part 3.

### 5.1 Label smoothing

A one-hot target asks the model to put probability 1 on the correct token, which pushes the logits toward infinity and makes the model over-confident. **Label smoothing** with $\epsilon_{ls} = 0.1$ trains against a softened target instead: $1 - \epsilon$ on the correct token and $\epsilon$ spread over the other tokens. The paper: *"This hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score."*

We implement it as in The Annotated Transformer: build the smoothed target distribution $q$ explicitly and minimise the KL divergence $\mathrm{KL}(q \,\|\, p)$ to the model's distribution $p$. The mass $\epsilon$ is spread over $V - 2$ tokens, since `<pad>` and the correct token are excluded, and rows whose target is `<pad>` contribute nothing.

In [ ]:
class LabelSmoothing(nn.Module):
    # KL(q || p) with q = 1 - eps on the correct token and eps / (V - 2) on every other non-pad token.
    def __init__(self, vocab_size, padding_idx=PAD, smoothing=0.1):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction="sum")
        self.vocab_size, self.padding_idx = vocab_size, padding_idx
        self.smoothing, self.confidence = smoothing, 1.0 - smoothing
        self.keep_target, self.true_dist = False, None       # set keep_target = True to inspect q

    def forward(self, logprobs, target):                     # logprobs: (N, V) log-probabilities; target: (N,)
        assert logprobs.size(1) == self.vocab_size
        true_dist = torch.full_like(logprobs, self.smoothing / (self.vocab_size - 2))
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0                   # never predict <pad>
        true_dist[target == self.padding_idx] = 0            # padded positions contribute nothing
        if self.keep_target:
            self.true_dist = true_dist.detach()
        return self.criterion(logprobs, true_dist)           # summed over tokens

# The smoothed target for 5 positions over a toy vocabulary of 7 tokens (id 0 = <pad>)
ls = LabelSmoothing(7, padding_idx=0, smoothing=0.4)
ls.keep_target = True
targets = torch.tensor([2, 1, 0, 3, 3])                      # position 2 is padding
ls(torch.log_softmax(torch.randn(5, 7), -1), targets)
fig, ax = plt.subplots(figsize=(4.2, 3))
ax.imshow(ls.true_dist, cmap=SEQ_CMAP); ax.grid(False)
ax.set_xlabel("vocabulary id"); ax.set_ylabel("position"); ax.set_title("smoothed targets, eps = 0.4")
plt.show()
print(ls.true_dist)

In [ ]:
# Relation to PyTorch's built-in: F.cross_entropy(..., label_smoothing=eps) spreads eps over ALL V classes
# (including the correct one and <pad>) and returns a cross-entropy, not a KL divergence.
torch.manual_seed(0)
logits = torch.randn(6, 50)
target = torch.tensor([5, 9, 0, 12, 7, 3])                  # one <pad> row
logprobs = logits.log_softmax(-1)
ls = LabelSmoothing(50, padding_idx=0, smoothing=0.1)
ls.keep_target = True
kl = ls(logprobs, target)
q = ls.true_dist
ce_q = -(q * logprobs).sum()                                 # cross-entropy against the same q
entropy_q = -(q[q > 0] * q[q > 0].log()).sum()
print(f"KL(q || p) = {kl:.4f} | CE(q, p) = {ce_q:.4f} | H(q) = {entropy_q:.4f} | CE - H = {ce_q - entropy_q:.4f}")
assert torch.allclose(ce_q - entropy_q, kl, atol=1e-4)       # KL = CE - H(q): same gradients, constant offset
builtin = F.cross_entropy(logits, target, ignore_index=0, label_smoothing=0.1, reduction="sum")
print(f"F.cross_entropy(label_smoothing=0.1) = {builtin:.4f}  (a slightly different q, so a slightly different number)")
print(f"plain NLL (no smoothing)             = {F.nll_loss(logprobs, target, ignore_index=0, reduction='sum'):.4f}")

In [ ]:
# Smoothing penalises over-confidence: the loss is lowest when the model gives the correct token
# about 1 - eps of the probability, not all of it. (Model: probability x on the correct token,
# the rest spread evenly over the other V - 1 tokens.)
Vtoy = 1000
xs = np.linspace(0.02, 0.999, 400)
fig, ax = plt.subplots(figsize=(6.5, 3.2))
for eps in (0.0, 0.1):
    crit = LabelSmoothing(Vtoy, padding_idx=0, smoothing=eps)
    losses = []
    for x in xs:
        p = torch.full((1, Vtoy), (1 - x) / (Vtoy - 1))
        p[0, 1] = x
        losses.append(crit(p.log(), torch.tensor([1])).item())
    best = xs[int(np.argmin(losses))]
    ax.plot(xs, losses, label=f"eps = {eps}: minimum at p = {best:.2f}")
ax.set_xlabel("probability the model gives the correct token"); ax.set_ylabel("loss")
ax.set_title("Label smoothing moves the optimum below 1"); ax.legend(); plt.show()

### 5.2 Adam and the warmup schedule

The paper uses Adam with $\beta_1 = 0.9$, $\beta_2 = 0.98$, $\epsilon = 10^{-9}$ and this learning rate (equation 3):

$$lrate = d_\text{model}^{-0.5} \cdot \min\!\left(step^{-0.5},\; step \cdot warmup^{-1.5}\right)$$

It rises linearly for `warmup` steps (4,000 in the paper), then decays as $1/\sqrt{step}$. The warmup matters most for post-norm Transformers: early in training Adam's second-moment estimates are noisy and the residual stream is fragile, and full-size steps can wreck the model before it learns anything. The $d_\text{model}^{-0.5}$ factor gives wider models smaller steps.

**Our setting differs from the paper in two ways, and both matter.**

1. We take far fewer steps (about 250 per epoch on Multi30k), so the warmup is shortened to 2,000 steps (about 8 epochs).
2. Our batches are about 12× smaller (2,048 tokens per side instead of ~25,000), so every gradient is much noisier.

In the formula, a shorter warmup *raises* the peak: with 2,000 steps it reaches 9.9e-4, above the paper's 7e-4, while our noisier gradients call for a *lower* one. Post-norm Transformers are known to be sensitive to the peak learning rate and the warmup length (Xiong et al., 2020). The first full run of this notebook used that 9.9e-4 peak, and in epoch 18 its training diverged: validation loss jumped from 2.6 to 5.0 and never recovered. (Its BLEU had already stalled at about 15 for a different reason, the initialisation problem described in 4.9.)

So we keep the formula's shape and scale it with `factor` so that it peaks at **4e-4**, in line with small-batch post-norm recipes (fairseq's IWSLT'14 German→English recipe peaks at 5e-4 with 4,096-token batches). `factor = 1` is the paper's formula.

In [ ]:
def noam_rate(step, d_model, warmup, factor=1.0):
    step = max(step, 1)
    return factor * d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)

def factor_for_peak(peak_lr, d_model, warmup):
    # The factor that makes the schedule peak (at step = warmup) at peak_lr
    return peak_lr / noam_rate(warmup, d_model, warmup)

def make_optimizer(model, d_model, warmup, factor=1.0):
    optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9,
                                 fused=device.type == "cuda")
    # LambdaLR multiplies the base lr (1.0) by our function of the step count
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: noam_rate(step + 1, d_model, warmup, factor))
    return optimizer, scheduler

steps = np.arange(1, 20001)
ours = factor_for_peak(4e-4, 512, 2000)
fig, ax = plt.subplots(figsize=(7, 3.2))
for d, w, f, label in ((512, 4000, 1.0, "paper: d_model 512, warmup 4000"),
                       (512, 8000, 1.0, "d_model 512, warmup 8000"),
                       (256, 4000, 1.0, "d_model 256, warmup 4000"),
                       (512, 2000, 1.0, "d_model 512, warmup 2000, factor 1 (too high for us)"),
                       (512, 2000, ours, f"ours: d_model 512, warmup 2000, factor {ours:.2f}")):
    ax.plot(steps, [noam_rate(s, d, w, f) for s in steps], label=label)
ax.set_xlabel("step"); ax.set_ylabel("learning rate"); ax.set_title("Warmup, then 1/sqrt(step) decay")
ax.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=8)
plt.show()
print(f"peak learning rate: paper base {noam_rate(4000, 512, 4000):.2e} | warmup 2000 with factor 1: "
      f"{noam_rate(2000, 512, 2000):.2e} | ours: {noam_rate(2000, 512, 2000, ours):.2e} (factor {ours:.3f})")

### 5.3 The training loop

One function runs an epoch in either mode. In training mode it does forward → loss → backward → optimizer step → schedule step for each batch; with no optimizer it only evaluates. It reports four numbers, and it's worth knowing which is which:

- **loss**: the label-smoothed objective we optimise (per token)
- **NLL**: plain negative log-likelihood of the correct tokens (per token), without smoothing
- **perplexity** = $e^{\text{NLL}}$: "as uncertain as a uniform choice among this many tokens"
- **accuracy**: how often the most likely next token is the correct one, with teacher forcing

The loss is divided by the number of real target tokens, so long and short batches weigh the same per token. Gradient clipping at norm 1.0 is **not** in the paper; it's a cheap safety net, and you can set `clip_grad=None` to train exactly as published.

In [ ]:
@dataclass
class EpochStats:
    loss: float       # label-smoothed objective per token
    nll: float        # negative log-likelihood per token
    ppl: float        # exp(nll)
    acc: float        # next-token accuracy under teacher forcing
    tokens: int
    seconds: float

    @property
    def tok_per_s(self):
        return self.tokens / max(self.seconds, 1e-9)

def run_epoch(model, batches, criterion, optimizer=None, scheduler=None, clip_grad=None, lr_log=None):
    training = optimizer is not None
    model.train(training)
    totals = Counter()
    start = time.time()
    for batch in batches:
        batch = batch.to(device)
        with torch.set_grad_enabled(training):
            with amp():
                hidden = model(batch.src, batch.tgt_in, batch.src_mask, batch.tgt_mask)   # (B, T, d_model)
                logprobs = model.generator(hidden)                                          # (B, T, V)
            logprobs = logprobs.float().reshape(-1, logprobs.size(-1))                      # (B*T, V)
            target = batch.tgt_out.reshape(-1)                                              # (B*T,)
            loss = criterion(logprobs, target)
            if training:
                (loss / batch.ntokens).backward()
                if clip_grad:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                if scheduler is not None:
                    scheduler.step()
                if lr_log is not None:
                    lr_log.append(optimizer.param_groups[0]["lr"])
        with torch.no_grad():
            real = target != PAD
            lp, tg = logprobs[real], target[real]
            totals["loss"] += loss.item()
            totals["nll"] += -lp.gather(1, tg[:, None]).sum().item()
            totals["correct"] += (lp.argmax(-1) == tg).sum().item()
            totals["tokens"] += batch.ntokens
    n = max(totals["tokens"], 1)
    return EpochStats(totals["loss"] / n, totals["nll"] / n, math.exp(min(totals["nll"] / n, 50)),
                      totals["correct"] / n, int(totals["tokens"]), time.time() - start)

# Sanity check before any training: an untrained model is close to uniform over V tokens, so its NLL
# should be close to ln(V). Its logits are random with variance ~1, which adds about 1/2 on top.
seed_everything(0)
check_model = make_model(ModelConfig(V, V, N=2, d_model=128, d_ff=256, h=4)).to(device)
stats = run_epoch(check_model, [collate(encoded["val"], list(range(64)))], LabelSmoothing(V, PAD, 0.1))
print(f"untrained: NLL {stats.nll:.2f} vs ln(V) + 0.5 = {math.log(V) + 0.5:.2f} | "
      f"accuracy {stats.acc:.4f} vs 1/V = {1 / V:.4f}")
del check_model

In [ ]:
CKPT_DIR = pathlib.Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

def state_on_cpu(model):
    return {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}

def save_checkpoint(path, model, state=None, **extra):
    # The tokenizer is saved too: token ids only mean something with the vocabulary the model was trained on.
    torch.save({"cfg": asdict(model.cfg), "state_dict": state or state_on_cpu(model),
                "tokenizer": tok.to_str(), **extra}, path)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location="cpu")    # works for a checkpoint written on any device
    model = make_model(ModelConfig(**ckpt["cfg"]))
    model.load_state_dict(ckpt["state_dict"])
    return model.to(device), ckpt

## Part 6 · Warm-up on synthetic data: copy and reverse

Before translating real sentences, train on tasks with a known answer. If the model can't learn to copy a sequence, something is broken, and a synthetic task tells you within seconds. The Annotated Transformer uses a copy task; we add a **reverse** task, because its attention pattern is easy to predict: to write output position $t$, the decoder should look at input position $L - 1 - t$, which draws an anti-diagonal.

Vocabulary: `0 <pad>`, `1 <s>`, `2 </s>` and ten symbols `3…12`. Sequences have random lengths from 4 to 12, so the padding masks are exercised too.

In [ ]:
SYN_VOCAB = 13

def synthetic_batch(batch_size, task, generator, min_len=4, max_len=12):
    srcs, tgts = [], []
    for _ in range(batch_size):
        length = int(torch.randint(min_len, max_len + 1, (1,), generator=generator))
        seq = torch.randint(3, SYN_VOCAB, (length,), generator=generator)
        body = seq if task == "copy" else seq.flip(0)
        srcs.append(torch.cat([seq, torch.tensor([EOS])]))
        tgts.append(torch.cat([torch.tensor([BOS]), body, torch.tensor([EOS])]))
    return Batch(pad_sequence(srcs, batch_first=True, padding_value=PAD),
                 pad_sequence(tgts, batch_first=True, padding_value=PAD))

demo = synthetic_batch(3, "reverse", torch.Generator().manual_seed(0))
print("src:\n", demo.src, "\ntgt_in:\n", demo.tgt_in, "\ntgt_out:\n", demo.tgt_out)

### 6.1 Greedy decoding

At inference time there is no target to feed in. The decoder generates **autoregressively**:

1. Encode the source once; the memory is reused at every step.
2. Start the output with `<s>`.
3. Run the decoder on the output so far and take the distribution at the **last** position.
4. Append the most likely token (greedy choice) and repeat until `</s>` or a length limit.

The batched version below decodes many sentences at once; sentences that have finished keep receiving `<pad>`. Each step re-runs the decoder over the whole prefix, so a sentence of length $T$ costs $O(T^2)$ decoder positions. Production decoders cache each layer's keys and values instead (the KV cache from your project's Day 1). We keep the simple version because it's easier to trace.

In [ ]:
@torch.no_grad()
def greedy_decode(model, src, src_mask, max_len, bos=BOS, eos=EOS, pad=PAD):
    model.eval()
    memory = model.encode(src, src_mask)                                     # (B, S, d_model), computed once
    ys = torch.full((src.size(0), 1), bos, dtype=torch.long, device=src.device)
    finished = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    for _ in range(max_len - 1):
        tgt_mask = subsequent_mask(ys.size(1), src.device)[None, None]        # (1, 1, t, t)
        hidden = model.decode(memory, src_mask, ys, tgt_mask)                 # (B, t, d_model)
        next_token = model.generator(hidden[:, -1]).argmax(-1)                # (B,): only the last position is new
        next_token = next_token.masked_fill(finished, pad)
        ys = torch.cat([ys, next_token[:, None]], dim=1)
        finished |= next_token == eos
        if finished.all():
            break
    return ys                                                                 # (B, t), starts with <s>

@torch.no_grad()
def trace_greedy(model, src, id_to_str, max_steps=30):
    # Print every decoding step: what the decoder reads, its top-3 guesses, and the pick.
    model.eval()
    src_mask = (src != PAD)[:, None, None, :]
    memory = model.encode(src, src_mask)
    ys = torch.tensor([[BOS]], device=src.device)
    for step in range(max_steps):
        hidden = model.decode(memory, src_mask, ys, subsequent_mask(ys.size(1), src.device)[None, None])
        top = model.generator(hidden[:, -1]).exp()[0].topk(3)
        guesses = ", ".join(f"{id_to_str(i)} {p:.2f}" for p, i in zip(top.values.tolist(), top.indices.tolist()))
        pick = top.indices[0].item()
        print(f"step {step:>2} | reads {' '.join(id_to_str(i) for i in ys[0].tolist()):<40} | top-3: {guesses:<32} -> {id_to_str(pick)}")
        ys = torch.cat([ys, torch.tensor([[pick]], device=src.device)], dim=1)
        if pick == EOS:
            break

def cut_at_eos(seq):
    seq = seq[1:] if seq and seq[0] == BOS else seq
    return seq[: seq.index(EOS)] if EOS in seq else seq

@torch.no_grad()
def synthetic_exact_match(model, task, n=512, seed=123):
    batch = synthetic_batch(n, task, torch.Generator().manual_seed(seed)).to(device)
    ys = greedy_decode(model, batch.src, batch.src_mask, max_len=batch.tgt.size(1) + 2)
    return sum(cut_at_eos(p) == cut_at_eos(r) for p, r in zip(ys.tolist(), batch.tgt.tolist())) / n

In [ ]:
def train_synthetic(task, steps, batch_size=128, chunk=250, seed=0):
    seed_everything(seed)
    cfg = ModelConfig(SYN_VOCAB, SYN_VOCAB, N=2, d_model=128, d_ff=256, h=4, dropout=0.1)
    model = make_model(cfg).to(device)
    criterion = LabelSmoothing(SYN_VOCAB, PAD, smoothing=0.0)       # exact targets: we want perfect copies
    optimizer, scheduler = make_optimizer(model, cfg.d_model, warmup=400)
    gen = torch.Generator().manual_seed(seed)
    log = []
    for done in range(0, steps, chunk):
        n = min(chunk, steps - done)
        stats = run_epoch(model, [synthetic_batch(batch_size, task, gen) for _ in range(n)],
                          criterion, optimizer, scheduler)
        exact = synthetic_exact_match(model, task)
        log.append((done + n, stats.loss, exact))
        print(f"{task:>7} step {done + n:>5} | loss {stats.loss:.4f} | token accuracy {stats.acc:.3f} | "
              f"exact match (greedy) {exact:.1%} | {stats.tok_per_s:,.0f} tokens/s")
    return model, log

SYN_STEPS = 150 if QUICK else 3000
copy_model, copy_log = train_synthetic("copy", SYN_STEPS)

In [ ]:
digit = lambda i: {PAD: "<pad>", BOS: "<s>", EOS: "</s>"}.get(i, str(i - 3))
example = synthetic_batch(1, "copy", torch.Generator().manual_seed(7))
print("source:", " ".join(digit(i) for i in example.src[0].tolist()))
trace_greedy(copy_model, example.src.to(device), digit)

In [ ]:
reverse_model, reverse_log = train_synthetic("reverse", SYN_STEPS)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for name, log in (("copy", copy_log), ("reverse", reverse_log)):
    axes[0].plot([s for s, _, _ in log], [l for _, l, _ in log], marker="o", label=name)
    axes[1].plot([s for s, _, _ in log], [a for _, _, a in log], marker="o", label=name)
axes[0].set_yscale("log"); axes[0].set_title("training loss"); axes[1].set_title("exact match, greedy decoding")
for ax in axes:
    ax.set_xlabel("step"); ax.legend()
plt.tight_layout(); plt.show()

**Looking inside.** To see where the decoder looks, we switch the attention modules to the explicit path that stores their weights, run one teacher-forced forward pass, and plot the **cross-attention** of the last decoder layer. Rows are output positions (the token being predicted); columns are source positions. For the reverse task, expect an anti-diagonal: output 0 attends to the last input symbol, and so on.

In [ ]:
def set_store_attention(model, flag):
    for m in model.modules():
        if isinstance(m, MultiHeadAttention):
            m.store_attn = flag
            if not flag:
                m.attn = None

def plot_heads(maps, row_labels, col_labels, title, max_heads=8, ncols=4, cell=2.7, panel_names=None):
    # maps: (heads, rows, cols) attention weights, one panel per head
    n = min(maps.size(0), max_heads)
    panel_names = panel_names or [f"head {i}" for i in range(n)]
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(cell * ncols, cell * nrows + 0.6), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.grid(False)
        if i >= n:
            ax.axis("off"); continue
        ax.imshow(maps[i], cmap=SEQ_CMAP, vmin=0, aspect="auto")
        ax.set_title(panel_names[i], fontsize=9)
        ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=7)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

example = synthetic_batch(1, "reverse", torch.Generator().manual_seed(3)).to(device)
reverse_model.eval()
set_store_attention(reverse_model, True)
with torch.no_grad():
    reverse_model(example.src, example.tgt_in, example.src_mask, example.tgt_mask)
cross = reverse_model.decoder.layers[-1].src_attn.attn[0]                    # (h, T, S)
set_store_attention(reverse_model, False)
plot_heads(cross, [digit(i) for i in example.tgt_out[0].tolist()], [digit(i) for i in example.src[0].tolist()],
           "reverse task: last decoder layer's cross-attention (rows = output, columns = input)")

**Check yourself**

- Why do the copy and reverse tasks use no label smoothing?
- The reverse model must know *where* each input symbol is. Which component gives it that information?
- Remove the causal mask (make `subsequent_mask` return all `True`) and retrain the copy model. Training loss collapses toward 0, but greedy decoding fails. Why?

## Part 7 · Training on Multi30k (German → English)

**Configurations.** `base` is the paper's base model. `small` (3 layers, `d_model` 256) trains about four times faster and is useful for experiments. `tiny` is only for quick mode.

Our batches hold about 2,048 tokens per side (the paper used ~25,000 per side on 8 GPUs). With ~29k sentence pairs that's about 250 steps per epoch; the training cell prints the exact number. The learning rate peaks at 4e-4 (see 5.2 for why not the formula's 9.9e-4). The cells below train, then compare three ways of picking the final weights.

**Watching for collapse.** Each epoch also prints the **cross-attention spread**: the entropy of the decoder's attention over the source, divided by its maximum, $\ln(\text{number of source tokens})$, averaged over decoder layers, heads and positions. 1 means uniform: no source word is preferred. It starts around 0.9 and should fall as the model learns which source words each output word depends on (in a smaller test model it went from 0.90 to 0.67 in the first 500 steps). If it climbs to 1.00 and stays there, the decoder is ignoring *where* things are in the source and translating from a blurred average of the sentence. BLEU then stalls and falls off with sentence length, which is what happened in the first full run of this notebook (see 4.9 and 5.2).

Re-running the notebook: with `REUSE_CHECKPOINT = True`, a finished run is loaded from `checkpoints/` instead of retrained, as long as it was trained with the same `TrainConfig` and tokenizer.

In [ ]:
PRESETS = {
    "base":  dict(N=6, d_model=512, d_ff=2048, h=8),     # the paper's base model
    "small": dict(N=3, d_model=256, d_ff=1024, h=8),     # faster, for experiments
    "tiny":  dict(N=2, d_model=64,  d_ff=128,  h=4),     # quick-mode smoke test
}

@dataclass
class TrainConfig:
    preset: str = "base"
    epochs: int = 25
    max_tokens: int = 2048        # per batch, per side (padded)
    warmup: int = 2000            # steps of linear warmup (about 8 epochs here)
    peak_lr: float = 4e-4         # where the schedule peaks; the formula alone would give 9.9e-4 (see 5.2)
    dropout: float = 0.1
    label_smoothing: float = 0.1
    clip_grad: float = 1.0        # not in the paper; None reproduces it exactly
    bleu_every: int = 1           # greedy BLEU on validation sentences every k epochs
    bleu_sentences: int = 500
    average_last: int = 5         # the paper averages the last 5 checkpoints for base models
    seed: int = 0

TCFG = TrainConfig()
if QUICK:
    TCFG = TrainConfig(preset="tiny", epochs=2, max_tokens=1024, warmup=100, peak_lr=1e-3, bleu_sentences=50, average_last=2)
steps_per_epoch = len(make_batches(encoded["train"], TCFG.max_tokens))
print(TCFG)
print(f"{steps_per_epoch} steps per epoch, {steps_per_epoch * TCFG.epochs:,} in total; the learning rate peaks at "
      f"{TCFG.peak_lr:.1e} after {TCFG.warmup:,} steps (epoch {TCFG.warmup / steps_per_epoch:.1f})")

In [ ]:
def ids_to_text(ids):
    ids = [i for i in ids if i not in (BOS, PAD)]
    if EOS in ids:
        ids = ids[: ids.index(EOS)]
    return tok.decode(ids)

@torch.no_grad()
def translate(model, sentences, method="greedy", batch_size=128, beam_size=4, alpha=0.6, max_extra=50):
    # Translate raw source strings. Greedy runs in batches; beam search (Part 8) runs per sentence.
    model.eval()
    out = [None] * len(sentences)
    order = sorted(range(len(sentences)), key=lambda i: len(sentences[i]))   # similar lengths together
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        ids = [tok.encode(sentences[i]).ids[: MAX_LEN - 1] + [EOS] for i in idx]
        src = pad_sequence([torch.tensor(x) for x in ids], batch_first=True, padding_value=PAD).to(device)
        src_mask = (src != PAD)[:, None, None, :]
        with amp():
            if method == "greedy":
                ys = greedy_decode(model, src, src_mask, max_len=src.size(1) + max_extra)   # paper: input length + 50
                for j, i in enumerate(idx):
                    out[i] = ids_to_text(ys[j].tolist())
            else:
                for j, i in enumerate(idx):
                    n = len(ids[j])
                    best = beam_search(model, src[j:j + 1, :n], src_mask[j:j + 1, ..., :n], beam_size, alpha, max_extra)
                    out[i] = ids_to_text(best)
    return out

def val_bleu(model, n=None, method="greedy"):
    pairs = raw["val"][:n] if n else raw["val"]
    hyps = translate(model, [s for s, _ in pairs], method=method)
    return sacrebleu.corpus_bleu(hyps, [[t for _, t in pairs]]).score

In [ ]:
@torch.no_grad()
def cross_attention_spread(model, batch):
    # Entropy of the decoder's attention over the source divided by its maximum, ln(number of source
    # tokens), averaged over decoder layers, heads and real target positions.
    # 1 = uniform (no source word preferred); lower = each output word looks at a few source words.
    model.eval()
    modules = [layer.src_attn for layer in model.decoder.layers]
    for m in modules:
        m.store_attn = True
    with amp():
        model(batch.src, batch.tgt_in, batch.src_mask, batch.tgt_mask)
    max_entropy = (batch.src != PAD).sum(-1).clamp_min(2).float().log().cpu()[:, None, None]   # (B, 1, 1)
    real = (batch.tgt_in != PAD).cpu()[:, None, :]                                               # (B, 1, T)
    spreads = []
    for m in modules:
        p = m.attn                                                     # (B, h, T, S), stored on the CPU
        entropy = -(p * p.clamp_min(1e-12).log()).sum(-1)              # (B, h, T)
        spreads.append((entropy / max_entropy)[real.expand_as(entropy)].mean())
        m.store_attn, m.attn = False, None
    return torch.stack(spreads).mean().item()

def train_translation(tcfg, model_overrides=None, label="", verbose=True):
    seed_everything(tcfg.seed)
    kwargs = {**PRESETS[tcfg.preset], "dropout": tcfg.dropout, **(model_overrides or {})}
    model = make_model(ModelConfig(V, V, **kwargs)).to(device)
    criterion = LabelSmoothing(V, PAD, tcfg.label_smoothing)
    factor = factor_for_peak(tcfg.peak_lr, model.cfg.d_model, tcfg.warmup)
    optimizer, scheduler = make_optimizer(model, model.cfg.d_model, tcfg.warmup, factor)
    val_batches = [collate(encoded["val"], b) for b in make_batches(encoded["val"], tcfg.max_tokens, shuffle=False)]
    probe = collate(encoded["val"], list(range(64))).to(device)    # fixed sentences for the spread measurement
    history, lrs, last_states = [], [], []
    best = dict(bleu=-1.0, epoch=0, state=None)
    best_val_loss = float("inf")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    if verbose:
        print(f"{label}{tcfg.preset}: {count_params(model) / 1e6:.1f}M parameters | {model.cfg}")
        print(f"cross-attention spread before training: {cross_attention_spread(model, probe):.2f}")
    for epoch in range(1, tcfg.epochs + 1):
        batches = (collate(encoded["train"], b) for b in
                   make_batches(encoded["train"], tcfg.max_tokens, shuffle=True, seed=tcfg.seed + epoch))
        train = run_epoch(model, batches, criterion, optimizer, scheduler, tcfg.clip_grad, lr_log=lrs)
        val = run_epoch(model, val_batches, criterion)
        spread = cross_attention_spread(model, probe)
        do_bleu = epoch % tcfg.bleu_every == 0 or epoch == tcfg.epochs
        bleu = val_bleu(model, tcfg.bleu_sentences) if do_bleu else float("nan")
        history.append(dict(epoch=epoch, train_loss=train.loss, val_loss=val.loss, val_nll=val.nll,
                            val_ppl=val.ppl, val_acc=val.acc, val_bleu=bleu, cross_spread=spread,
                            tok_per_s=train.tok_per_s, seconds=train.seconds, step=len(lrs)))
        if verbose:
            print(f"epoch {epoch:>2} | step {len(lrs):>5} | lr {lrs[-1]:.2e} | train loss {train.loss:.3f} | "
                  f"val loss {val.loss:.3f} | val ppl {val.ppl:6.2f} | val acc {val.acc:.3f} | "
                  f"val BLEU {bleu:5.2f} | cross-attn spread {spread:.2f} | "
                  f"{train.tok_per_s / 1e3:5.1f}k tok/s | {train.seconds:4.1f}s")
            if val.loss > best_val_loss + 1.0:
                print(f"         warning: validation loss is {val.loss - best_val_loss:.1f} above its best: training has diverged")
        best_val_loss = min(best_val_loss, val.loss)
        if epoch > tcfg.epochs - tcfg.average_last:
            last_states.append(state_on_cpu(model))
        if do_bleu and bleu > best["bleu"]:
            best = dict(bleu=bleu, epoch=epoch, state=state_on_cpu(model))
    if verbose and device.type == "cuda":
        print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
    return model, history, lrs, best, last_states

def average_states(states):
    # Checkpoint averaging (section 6.1): the element-wise mean of the last k sets of weights.
    return {k: sum(s[k].float() for s in states) / len(states) for k in states[0]}

In [ ]:
REUSE_CHECKPOINT = True
CKPT_PATH = CKPT_DIR / f"multi30k_{SRC_LANG}-{TGT_LANG}_{TCFG.preset}_v{V}{'_quick' if QUICK else ''}.pt"

model = None
if REUSE_CHECKPOINT and CKPT_PATH.exists():
    model, ckpt = load_checkpoint(CKPT_PATH)
    if ckpt.get("train_cfg") != asdict(TCFG) or ckpt.get("tokenizer") != tok.to_str():
        print(f"{CKPT_PATH} was trained with other settings or another tokenizer, so we train again")
        model = None
if model is not None:
    history, lrs, variant_bleu, chosen = ckpt["history"], ckpt["lrs"], ckpt["variant_bleu"], ckpt["chosen"]
    print(f"loaded {CKPT_PATH} ({chosen}); delete the file or set REUSE_CHECKPOINT = False to retrain")
else:
    t0 = time.time()
    model, history, lrs, best, last_states = train_translation(TCFG)
    print(f"training took {(time.time() - t0) / 60:.1f} min")

    # Three candidates (explained below the plots); we pick by BLEU on the full validation set
    candidates = {"final epoch": state_on_cpu(model),
                  f"best validation epoch ({best['epoch']})": best["state"],
                  f"average of last {len(last_states)} epochs": average_states(last_states)}
    variant_bleu = {}
    for name, state in candidates.items():
        model.load_state_dict(state)
        variant_bleu[name] = val_bleu(model)
        print(f"{name:<32} validation BLEU (greedy, all {len(raw['val'])} sentences) = {variant_bleu[name]:.2f}")
    chosen = max(variant_bleu, key=variant_bleu.get)
    model.load_state_dict(candidates[chosen])
    save_checkpoint(CKPT_PATH, model, train_cfg=asdict(TCFG), history=history, lrs=lrs,
                    variant_bleu=variant_bleu, chosen=chosen)
    print(f"selected: {chosen} -> saved {CKPT_PATH}")

In [ ]:
epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.4))
axes[0, 0].plot(epochs, [h["train_loss"] for h in history], marker="o", label="train")
axes[0, 0].plot(epochs, [h["val_loss"] for h in history], marker="o", label="validation")
axes[0, 0].set_title("label-smoothed loss per token"); axes[0, 0].legend()
axes[0, 1].plot(epochs, [h["val_ppl"] for h in history], marker="o")
axes[0, 1].set_title("validation perplexity (no smoothing)")
axes[0, 2].plot(epochs, [h["val_acc"] for h in history], marker="o")
axes[0, 2].set_title("validation next-token accuracy")
axes[1, 0].plot(epochs, [h["val_bleu"] for h in history], marker="o")
axes[1, 0].set_title(f"validation BLEU, greedy, first {TCFG.bleu_sentences} sentences")
axes[1, 1].plot(epochs, [h.get("cross_spread", float("nan")) for h in history], marker="o")
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].set_title("cross-attention spread (1 = uniform)")
axes[1, 2].plot(np.arange(1, len(lrs) + 1), lrs)
axes[1, 2].set_title("learning rate per step")
for ax in axes.flat[:5]:
    ax.set_xlabel("epoch"); ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[1, 2].set_xlabel("step")
plt.tight_layout(); plt.show()

### Picking the final weights: checkpoint averaging

The paper (section 6.1): *"For the base models, we used a single model obtained by averaging the last 5 checkpoints, which were written at 10-minute intervals."* Averaging weights from nearby points on the training trajectory cancels some of the noise of SGD and usually lands in a flatter, slightly better region of the loss surface. It's free at inference time: the result is one model.

The training cell compared three candidates on the full validation set: the final epoch, the epoch with the best validation BLEU, and the average of the last few epochs. It keeps the winner. Choosing on validation, never on test, keeps the test score honest.

In [ ]:
for name, score in variant_bleu.items():
    print(f"{name:<32} validation BLEU {score:6.2f}" + ("   <- selected" if name == chosen else ""))

## Part 8 · Inference: greedy decoding and beam search

Greedy decoding commits to the single most likely token at every step. A locally best word can lead into a poor sentence, and greedy can never take it back. **Beam search** keeps the $k$ best partial translations (the *beam*) at each step: it extends each by every possible next token, scores the extensions by total log-probability, keeps the top $k$, and moves a hypothesis to the finished list when it emits `</s>`.

Raw log-probability favours short outputs, because every extra token adds a negative term. The paper uses beam size 4 with the **length penalty** of Wu et al. (2016), $\alpha = 0.6$, and a maximum output length of input length + 50, terminating early when possible (section 6.1). Finished hypotheses are compared by

$$\text{score}(Y) = \frac{\log P(Y \mid X)}{lp(Y)}, \qquad lp(Y) = \left(\frac{5 + |Y|}{6}\right)^{\alpha}$$

**Early stopping**: alive hypotheses can only lose log-probability as they grow, so once the best finished score beats the best alive score divided by the largest possible $lp$, nothing alive can win, and we stop.

A common shortcut stops as soon as $k$ hypotheses have finished. It's faster but biased toward short outputs, because short hypotheses finish first. The first version of this notebook used it, and its beam translations came out 17% shorter than the references (brevity penalty 0.81).

In [ ]:
@torch.no_grad()
def beam_search(model, src, src_mask, beam_size=4, alpha=0.6, max_extra=50, bos=BOS, eos=EOS):
    # src: (1, S). Returns the best output token ids (without <s>).
    model.eval()
    memory = model.encode(src, src_mask)                                   # (1, S, d_model)
    max_len = src.size(1) + max_extra
    lp = lambda length: ((5 + length) / 6) ** alpha                        # GNMT length penalty
    hyps = torch.full((1, 1), bos, dtype=torch.long, device=src.device)    # alive hypotheses (k, t)
    scores = torch.zeros(1, device=src.device)                             # their total log-probabilities
    finished = []                                                          # (normalised score, token list)
    for t in range(1, max_len + 1):
        k = hyps.size(0)
        hidden = model.decode(memory.expand(k, -1, -1), src_mask.expand(k, -1, -1, -1), hyps,
                              subsequent_mask(t, src.device)[None, None])
        logp = model.generator(hidden[:, -1]).float()                     # (k, V)
        vocab = logp.size(-1)
        candidates = (scores[:, None] + logp).view(-1)                    # every hypothesis x every next token
        top_scores, top_ids = candidates.topk(min(2 * beam_size, candidates.numel()))
        new_hyps, new_scores = [], []
        for score, flat in zip(top_scores.tolist(), top_ids.tolist()):
            beam, token = divmod(flat, vocab)
            seq = hyps[beam].tolist() + [token]
            if token == eos:
                finished.append((score / lp(len(seq) - 1), seq))           # length excludes <s>
            else:
                new_hyps.append(seq); new_scores.append(score)
            if len(new_hyps) == beam_size:
                break
        if not new_hyps:                                                   # every candidate ended
            break
        hyps = torch.tensor(new_hyps, device=src.device)
        scores = torch.tensor(new_scores, device=src.device)
        if finished and max(f for f, _ in finished) > max(new_scores) / lp(max_len):
            break                                                          # no alive hypothesis can catch up
    else:                                                                  # hit the length limit:
        finished += [(s / lp(len(h) - 1), h) for s, h in zip(scores.tolist(), hyps.tolist())]   # unfinished ones compete too
    return max(finished, key=lambda f: f[0])[1][1:]

In [ ]:
rng = random.Random(5)
for src_text, ref in rng.sample(raw["test"], 5):
    t0 = time.time(); greedy = translate(model, [src_text])[0]; t_g = time.time() - t0
    t0 = time.time(); beam = translate(model, [src_text], method="beam")[0]; t_b = time.time() - t0
    print(f"{SRC_LANG.upper()}:     {src_text}\nREF:    {ref}\nGREEDY: {greedy}   ({t_g * 1000:.0f} ms)\n"
          f"BEAM 4: {beam}   ({t_b * 1000:.0f} ms)\n")

In [ ]:
# Watch the decoder work, one token at a time, on a real sentence
src_text = raw["test"][0][0]
print(src_text, "\n")
trace_greedy(model, torch.tensor([tok.encode(src_text).ids + [EOS]], device=device), tok.id_to_token)

In [ ]:
# Try your own German sentences (Multi30k is image captions, so describe a scene)
mine = ["Ein Mann fährt mit dem Fahrrad durch die Stadt.",
        "Zwei Kinder spielen im Schnee mit einem schwarzen Hund.",
        "Eine Frau mit einem roten Hut verkauft Blumen auf dem Markt."]
for s, g, b4 in zip(mine, translate(model, mine), translate(model, mine, method="beam")):
    print(f"{s}\n  greedy: {g}\n  beam:   {b4}\n")

## Part 9 · Evaluation

| Metric | Measures | Needs decoding? |
|---|---|---|
| perplexity, $e^{\text{NLL}}$ | how much probability the model gives the reference translation | no (teacher forcing) |
| token accuracy | how often the most likely next token is the reference token, given the reference prefix | no |
| **BLEU** | n-gram overlap between the model's own translation and the reference (the paper's metric) | yes |
| chrF | character n-gram F-score: more forgiving of morphology, often agrees better with people | yes |

Teacher-forced metrics are cheap and smooth, so they're good for watching training. BLEU and chrF judge the output you'd actually ship, including decoding errors that teacher forcing never sees.

### 9.1 Teacher-forced metrics on the test set

In [ ]:
criterion = LabelSmoothing(V, PAD, TCFG.label_smoothing)
test_batches = [collate(encoded["test"], b) for b in make_batches(encoded["test"], TCFG.max_tokens, shuffle=False)]
test_stats = run_epoch(model, test_batches, criterion)
print(f"test: label-smoothed loss {test_stats.loss:.3f} | NLL {test_stats.nll:.3f} | "
      f"perplexity {test_stats.ppl:.2f} | next-token accuracy {test_stats.acc:.1%}")

### 9.2 BLEU from scratch

BLEU (Papineni et al., 2002) for a whole test set:

1. **Clipped n-gram precision** $p_n$ for $n = 1 \ldots 4$: count the hypothesis n-grams that also appear in the reference, clipping each n-gram's count at its count in the reference (so "the the the" cannot score by repeating a matching word). Sum the matches and the totals over all sentences, then divide.
2. **Geometric mean** of $p_1 \ldots p_4$.
3. **Brevity penalty** $BP = \min\!\left(1,\; e^{\,1 - r/c}\right)$, with $c$ the total hypothesis length and $r$ the total reference length. Precision alone would reward translating only the easy words; BP punishes short output.

$$\text{BLEU} = 100 \cdot BP \cdot \exp\!\left(\tfrac{1}{4}\textstyle\sum_{n=1}^{4} \log p_n\right)$$

It's a **corpus** metric: the counts are pooled before dividing, which is not the same as averaging per-sentence BLEU. sacreBLEU, the standard implementation, also fixes the tokenization (`13a`), so scores are comparable across papers. We tokenize with the same `13a` rules and check that our number matches sacreBLEU's exactly.

In [ ]:
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a
tokenize_13a = Tokenizer13a()

def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def corpus_bleu_scratch(hypotheses, references, max_n=4):
    matches, totals = [0] * max_n, [0] * max_n
    hyp_len = ref_len = 0
    for hyp, ref in zip(hypotheses, references):
        h, r = tokenize_13a(hyp).split(), tokenize_13a(ref).split()
        hyp_len += len(h); ref_len += len(r)
        for n in range(1, max_n + 1):
            h_counts, r_counts = ngram_counts(h, n), ngram_counts(r, n)
            matches[n - 1] += sum(min(c, r_counts[g]) for g, c in h_counts.items())   # clipped matches
            totals[n - 1] += max(len(h) - n + 1, 0)
    precisions = [m / t if t else 0.0 for m, t in zip(matches, totals)]
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    bleu = 0.0 if min(precisions) == 0 else 100 * bp * math.exp(sum(math.log(p) for p in precisions) / max_n)
    return dict(bleu=bleu, precisions=[100 * p for p in precisions], bp=bp, hyp_len=hyp_len, ref_len=ref_len)

# A worked example: watch clipping and the brevity penalty
ref = "the cat is on the mat"
for hyp in ("the cat is on the mat", "the the the the the the", "the cat is on", "a cat is sitting on the mat"):
    s = corpus_bleu_scratch([hyp], [ref])
    print(f"{hyp!r:<32} BLEU {s['bleu']:6.2f} | p1..p4 {[round(p, 1) for p in s['precisions']]} | BP {s['bp']:.3f}")

Line by line: a perfect match scores 100. Repeating "the" earns only the 2 matches the reference allows (clipping), and no bigram matches, so BLEU is 0. A correct but short prefix has perfect precisions and is scaled down by the brevity penalty alone. The last line is a reasonable paraphrase that shares no 4-gram with the reference, so its BLEU is 0. That's why BLEU is pooled over a whole test set, where every $p_n$ is far from zero. For single sentences sacreBLEU smooths zero counts (`sentence_bleu`), which is why the two implementations can differ on one sentence but agree on a corpus.

Now the real check, on all 1,000 test translations:

In [ ]:
test_src = [s for s, _ in raw["test"]]
test_ref = [t for _, t in raw["test"]]
t0 = time.time()
greedy_hyps = translate(model, test_src)
greedy_seconds = time.time() - t0

ours = corpus_bleu_scratch(greedy_hyps, test_ref)
theirs = sacrebleu.corpus_bleu(greedy_hyps, [test_ref])
print(f"from scratch: BLEU {ours['bleu']:.2f}, precisions {[round(p, 1) for p in ours['precisions']]}, BP {ours['bp']:.3f}")
print(f"sacreBLEU:    {theirs}")
assert abs(ours["bleu"] - theirs.score) < 1e-6
print("identical")

### 9.3 The test-set table

Greedy versus beam search on all 1,000 test sentences. For reference, the paper reports 27.3 BLEU (base) and 28.4 (big) on WMT14 English→German; Multi30k sentences are short, formulaic captions, so scores here are much higher and not comparable.

In [ ]:
t0 = time.time()
beam_hyps = translate(model, test_src, method="beam")
beam_seconds = time.time() - t0

print(f"{'decoding':<10} {'BLEU':>6} {'chrF':>6} {'BP':>6} {'hyp/ref length':>15} {'seconds':>8}")
for name, hyps, secs in (("greedy", greedy_hyps, greedy_seconds), ("beam 4", beam_hyps, beam_seconds)):
    b = sacrebleu.corpus_bleu(hyps, [test_ref])
    c = sacrebleu.corpus_chrf(hyps, [test_ref])
    print(f"{name:<10} {b.score:>6.2f} {c.score:>6.2f} {b.bp:>6.3f} {b.sys_len / b.ref_len:>15.3f} {secs:>8.1f}")
changed = sum(g != b4 for g, b4 in zip(greedy_hyps, beam_hyps))
print(f"\nbeam search changed {changed} of {len(test_src)} translations")

In [ ]:
# Where does the model struggle? BLEU by source length, and the worst sentences
src_lens = np.array([len(tok.encode(s).ids) for s in test_src])
edges = [0, 10, 14, 18, 22, 100]
labels, scores, counts = [], [], []
for lo, hi in zip(edges, edges[1:]):
    idx = [i for i, n in enumerate(src_lens) if lo < n <= hi]
    if idx:
        labels.append(f"{lo + 1}–{hi}" if hi < 100 else f">{lo}")
        scores.append(sacrebleu.corpus_bleu([beam_hyps[i] for i in idx], [[test_ref[i] for i in idx]]).score)
        counts.append(len(idx))
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(labels, scores, color=SERIES[0], width=0.6)
for x, (s, n) in enumerate(zip(scores, counts)):
    ax.text(x, s + 0.02 * max(scores), f"{s:.1f}\n(n={n})", ha="center", va="bottom", fontsize=8, color="#52514e")
ax.set_xlabel("source length (tokens)"); ax.set_ylabel("BLEU"); ax.set_title("Test BLEU by sentence length (beam 4)")
ax.set_ylim(0, max(scores) * 1.25); plt.show()

sent_bleu = [sacrebleu.sentence_bleu(h, [r]).score for h, r in zip(beam_hyps, test_ref)]
for i in np.argsort(sent_bleu)[:3]:
    print(f"sentence BLEU {sent_bleu[i]:5.1f}\n  {SRC_LANG.upper()}:  {test_src[i]}\n  REF: {test_ref[i]}\n  OUT: {beam_hyps[i]}\n")

## Part 10 · Visualising attention

The trained model has $N \times h$ heads in each of its three attention types. We translate one sentence, then replay the forward pass with the attention weights stored, and plot:

- **encoder self-attention**: how source tokens mix with each other
- **decoder self-attention**: lower-triangular by construction (the causal mask)
- **cross-attention**: for each output token, which source tokens it looked at. Averaged over heads, this often resembles a word alignment.

Two things to look for: heads that attend to the previous or next token (positional heads), and heads that spread out evenly (often acting as "no-op" heads). Last, a summary over 200 sentences: the average **entropy** of each head's attention. Low entropy means sharp, focused attention; uniform attention over a 15-token sentence has entropy $\ln 15 \approx 2.7$ nats.

**Warning signs** of a broken model: cross-attention that is uniform in every layer (the decoder translates from a blurred average of the source), or entropy near 0 in almost every head (one-hot attention, often a sign of unstable training). The first full run of this notebook showed both (see 4.9).

In [ ]:
@torch.no_grad()
def attention_maps(model, src_text):
    model.eval()
    src_ids = tok.encode(src_text).ids + [EOS]
    src = torch.tensor([src_ids], device=device)
    src_mask = (src != PAD)[:, None, None, :]
    ys = greedy_decode(model, src, src_mask, max_len=len(src_ids) + 50)[0].tolist()
    ys = ys[: ys.index(EOS) + 1] if EOS in ys else ys
    tgt_in = torch.tensor([ys[:-1]], device=device)
    set_store_attention(model, True)
    try:
        model(src, tgt_in, src_mask, subsequent_mask(tgt_in.size(1), device)[None, None])
        maps = dict(enc=[l.self_attn.attn[0] for l in model.encoder.layers],
                    dec=[l.self_attn.attn[0] for l in model.decoder.layers],
                    cross=[l.src_attn.attn[0] for l in model.decoder.layers])
    finally:
        set_store_attention(model, False)
    maps.update(src=[tok.id_to_token(i) for i in src_ids],
                reads=[tok.id_to_token(i) for i in ys[:-1]],      # what each decoder position reads
                writes=[tok.id_to_token(i) for i in ys[1:]],      # what it predicts
                translation=ids_to_text(ys))
    return maps

maps = attention_maps(model, "Zwei Männer stehen neben einem roten Auto auf der Straße.")
print("translation:", maps["translation"])
n_layers = len(maps["enc"])
plot_heads(maps["enc"][0], maps["src"], maps["src"], "encoder self-attention, layer 1")
plot_heads(maps["enc"][-1], maps["src"], maps["src"], f"encoder self-attention, layer {n_layers}")

In [ ]:
plot_heads(maps["dec"][-1], maps["reads"], maps["reads"], f"decoder self-attention, layer {n_layers} (causal)")
plot_heads(maps["cross"][-1], maps["writes"], maps["src"],
           f"cross-attention, layer {n_layers}: rows = output token, columns = source token")

In [ ]:
# Cross-attention averaged over heads, one panel per decoder layer
mean_cross = torch.stack([m.mean(0) for m in maps["cross"]])          # (N, T, S)
plot_heads(mean_cross, maps["writes"], maps["src"], "cross-attention averaged over heads, by layer",
           max_heads=n_layers, ncols=min(n_layers, 3), cell=3.4,
           panel_names=[f"layer {i + 1}" for i in range(n_layers)])

In [ ]:
@torch.no_grad()
def attention_entropy(model, data, n=200, max_tokens=2048):
    # Mean entropy (nats) of each head's attention rows over the real query positions of n sentences.
    model.eval()
    layers = {"encoder self": [l.self_attn for l in model.encoder.layers],
              "decoder self": [l.self_attn for l in model.decoder.layers],
              "cross": [l.src_attn for l in model.decoder.layers]}
    sums = {k: torch.zeros(len(v), model.cfg.h) for k, v in layers.items()}
    counts = dict.fromkeys(layers, 0)
    set_store_attention(model, True)
    try:
        for idx in make_batches(data[:n], max_tokens, shuffle=False):
            b = collate(data, idx).to(device)
            model(b.src, b.tgt_in, b.src_mask, b.tgt_mask)
            real_q = {"encoder self": (b.src != PAD).cpu(), "decoder self": (b.tgt_in != PAD).cpu(),
                      "cross": (b.tgt_in != PAD).cpu()}
            for kind, mods in layers.items():
                for li, m in enumerate(mods):
                    p = m.attn                                                 # (B, h, T_q, T_k)
                    ent = -(p * p.clamp_min(1e-12).log()).sum(-1)              # (B, h, T_q)
                    sums[kind][li] += (ent * real_q[kind][:, None, :]).sum(dim=(0, 2))
                counts[kind] += int(real_q[kind].sum())
    finally:
        set_store_attention(model, False)
    return {k: sums[k] / counts[k] for k in sums}

entropy = attention_entropy(model, encoded["val"])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, (kind, ent) in zip(axes, entropy.items()):
    im = ax.imshow(ent, cmap=SEQ_CMAP, aspect="auto")
    ax.set_title(f"{kind}: mean entropy (nats)"); ax.set_xlabel("head"); ax.set_ylabel("layer"); ax.grid(False)
    ax.set_yticks(range(ent.size(0))); ax.set_yticklabels(range(1, ent.size(0) + 1))
    fig.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.show()

## Part 11 · Why self-attention: what attention costs

Table 1 of the paper compares layer types for a sequence of length $n$ and width $d$:

| Layer type | Complexity per layer | Sequential operations | Maximum path length |
|---|---|---|---|
| Self-attention | $O(n^2 \cdot d)$ | $O(1)$ | $O(1)$ |
| Recurrent | $O(n \cdot d^2)$ | $O(n)$ | $O(n)$ |
| Convolutional (kernel $k$) | $O(k \cdot n \cdot d^2)$ | $O(1)$ | $O(\log_k n)$ |
| Self-attention, restricted to a window $r$ | $O(r \cdot n \cdot d)$ | $O(1)$ | $O(n / r)$ |

The arguments for self-attention: **every pair of positions is one step apart** (short paths make long-range dependencies easier to learn), and **nothing is sequential across positions** (the whole sequence trains in parallel). The price is the $n^2$ term: every query is scored against every key. While $n < d$ (sentences of ~20 tokens, $d = 512$) that's cheaper than a recurrent layer. For long sequences it dominates, which is why FlashAttention-style fused kernels exist: they compute exactly the same result without ever storing the full $n \times n$ matrix.

The benchmark times one multi-head attention layer ($d_\text{model} = 512$, $h = 8$) as $n$ grows, with our explicit implementation (which stores the $n \times n$ weights) and with PyTorch's fused kernel.

In [ ]:
@torch.no_grad()
def time_attention(n, use_sdpa, d_model=512, h=8, reps=10):
    mha = MultiHeadAttention(d_model, h, dropout=0.0, use_sdpa=use_sdpa).to(device).eval()
    x = torch.randn(1, n, d_model, device=device)
    with amp():
        for _ in range(2):
            mha(x, x, x)                                   # warm-up
        sync()
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(reps):
            mha(x, x, x)
        sync()                                             # wait for the GPU before stopping the clock
    ms = (time.perf_counter() - t0) / reps * 1000
    mem = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else float("nan")
    return ms, mem

if QUICK:
    lengths = [64, 128, 256, 512]
elif device.type == "cuda":
    lengths = [128, 256, 512, 1024, 2048, 4096, 8192]
else:
    lengths = [128, 256, 512, 1024, 2048, 4096]           # at 8192 the explicit path holds ~4 GB of scores
results = {name: [time_attention(n, flag) for n in lengths] for name, flag in (("explicit (stores n x n)", False), ("fused SDPA", True))}
print(f"{'n':>6} " + " ".join(f"{name + ' ms':>26}" for name in results))
for i, n in enumerate(lengths):
    print(f"{n:>6} " + " ".join(f"{results[name][i][0]:>26.2f}" for name in results))

fig, axes = plt.subplots(1, 2 if device.type == "cuda" else 1, figsize=(11 if device.type == "cuda" else 5.5, 3.3), squeeze=False)
for name, res in results.items():
    axes[0, 0].plot(lengths, [r[0] for r in res], marker="o", label=name)
    if device.type == "cuda":
        axes[0, 1].plot(lengths, [r[1] for r in res], marker="o", label=name)
axes[0, 0].set_title("time per attention layer (ms)")
if device.type == "cuda":
    axes[0, 1].set_title("peak GPU memory (MiB)")
plain = plt.FuncFormatter(lambda v, _: f"{v:g}")             # 1024 rather than 2^10
for ax in axes.flat:
    ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xlabel("sequence length n"); ax.legend()
    ax.set_xticks(lengths); ax.xaxis.set_major_formatter(plain); ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.yaxis.set_major_locator(plt.LogLocator(subs=(1, 2, 5)))   # ticks at 1, 2, 5, 10, 20, 50, ...
    ax.yaxis.set_major_formatter(plain); ax.yaxis.set_minor_formatter(plt.NullFormatter())
plt.tight_layout(); plt.show()

## Part 12 · Ablations in the style of Table 3 (optional)

Table 3 of the paper varies one thing at a time on the base model. Its findings: one head is 0.9 BLEU worse than the best setting and too many heads also hurt (A); smaller $d_k$ hurts (B); bigger models are better and dropout matters (C, D); learned positional embeddings score nearly the same as sinusoids (E).

The cell below runs the same kind of experiment on the `small` model for a fixed number of epochs and reports validation BLEU (greedy) and perplexity. One short run can move by a BLEU point or so from the random seed alone, so treat small differences as noise and big ones as signal. Each variant is a full 12-epoch training run, so this cell takes a while. Set `RUN_ABLATIONS = True` to run it.

In [ ]:
RUN_ABLATIONS = QUICK          # set True to run the experiments below
VARIANTS = {                   # name: (model overrides, training-config overrides)
    "reference (small)":        ({}, {}),
    "(A) 1 head":               ({"h": 1}, {}),
    "(A) 32 heads":             ({"h": 32}, {}),
    "(D) no dropout":           ({}, {"dropout": 0.0}),
    "(D) no label smoothing":   ({}, {"label_smoothing": 0.0}),
    "(E) learned positions":    ({"pos_encoding": "learned"}, {}),
    "pre-norm":                 ({"norm_first": True}, {}),
    "no 1/sqrt(d_k) scaling":   ({"scale_attention": False}, {}),
    "unscaled residual init":   ({"scale_residual_init": False}, {}),
    "peak lr 9.9e-4 (formula)": ({}, {"peak_lr": 9.9e-4}),
}
if RUN_ABLATIONS:
    epochs = 1 if QUICK else 12
    base_cfg = TrainConfig(preset="tiny" if QUICK else "small", epochs=epochs, warmup=100 if QUICK else 1000,
                           max_tokens=1024 if QUICK else 2048, bleu_every=epochs,
                           bleu_sentences=50 if QUICK else len(raw["val"]), average_last=1)
    ablation = {}
    for name, (model_over, train_over) in VARIANTS.items():
        if QUICK and name not in ("reference (small)", "(A) 1 head"):
            continue
        t0 = time.time()
        _, hist, _, _, _ = train_translation(replace(base_cfg, **train_over), model_over, verbose=False)
        ablation[name] = (hist[-1]["val_bleu"], hist[-1]["val_ppl"], time.time() - t0)
        print(f"{name:<26} validation BLEU {ablation[name][0]:6.2f} | perplexity {ablation[name][1]:6.2f} | {ablation[name][2] / 60:4.1f} min")
    names = list(ablation)
    fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(names) + 1))
    ax.barh(names[::-1], [ablation[n][0] for n in names[::-1]], color=SERIES[0], height=0.6)
    top = max(v[0] for v in ablation.values())
    for y, n in enumerate(names[::-1]):
        ax.text(ablation[n][0] + 0.01 * top, y, f"{ablation[n][0]:.1f}", va="center", fontsize=8, color="#52514e")
    ax.set_xlim(0, top * 1.12)
    ax.set_xlabel("validation BLEU (greedy)")
    ax.set_title(f"Ablations, {epochs} epoch{'s' if epochs != 1 else ''} each"); plt.show()

## Part 13 · Recap

### From the paper to this notebook

| Paper | What it says | Where it lives here |
|---|---|---|
| 3.1 | encoder and decoder stacks, N = 6, residual + LayerNorm around each sub-layer | `Encoder`, `Decoder`, `SublayerConnection` (4.6–4.8) |
| 3.2.1 | scaled dot-product attention, $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ | `attention()` and the $\sqrt{d_k}$ experiment (4.3) |
| 3.2.2 | multi-head attention, h = 8, $d_k = d_v = 64$ | `MultiHeadAttention` (4.4) |
| 3.2.3 | three uses of attention; masking illegal connections with $-\infty$ | the `Batch` masks (Part 3), `DecoderLayer` (4.8) |
| 3.3 | position-wise feed-forward, $d_{ff} = 2048$ | `PositionwiseFeedForward` (4.5) |
| 3.4 | shared embeddings and output projection; embeddings × $\sqrt{d_\text{model}}$ | `Embeddings`, weight tying in `make_model` (4.1, 4.9) |
| 3.5 | sinusoidal positional encoding; learned alternative | `SinusoidalPositionalEncoding`, `LearnedPositionalEncoding` (4.2) |
| 4 | why self-attention: complexity, parallelism, path length | Part 11 |
| 5.1 | BPE, shared vocabulary, batches by length (~25k tokens) | Parts 2 and 3 |
| 5.3 | Adam (0.9, 0.98, 1e-9), warmup schedule | `make_optimizer`, `noam_rate` (5.2) |
| 5.4 | residual dropout 0.1; label smoothing 0.1 | `SublayerConnection`, `LabelSmoothing` (5.1) |
| 6.1 | checkpoint averaging; beam 4, α = 0.6, max length input + 50; BLEU | Part 7, `beam_search` (Part 8), Part 9 |
| 6.2 | Table 3 model variations | Part 12 |

### Exercises

1. **Flip the direction** to English→German (`SRC_LANG, TGT_LANG = "en", "de"`) and retrain. Which direction scores higher, and why might that be?
2. **KV cache for the decoder.** Cache each decoder layer's self-attention keys and values so each greedy step processes only the newest token. Check that the outputs are identical, then measure the speed-up as sentences get longer.
3. **Batched beam search.** Run beam search for a whole batch at once, with a `(B × k, t)` tensor of hypotheses. Check that it matches the per-sentence version.
4. **Post-norm without warmup.** Set `warmup=1` (and a smaller `lr_factor`) with `norm_first=False`, then with `norm_first=True`. Which one trains?
5. **Vocabulary size.** Retrain with 2,000 and 16,000 BPE tokens. Plot sentence length in tokens against BLEU.
6. **Alignment from attention.** For each output word, take the source token with the highest head-averaged cross-attention in the last layer. How often does it match an obvious word translation (Hund → dog)?
7. **Sampling.** Add top-p sampling to `greedy_decode`. Compare BLEU and output diversity with greedy and beam search.
8. **Big-model regularisation.** Train `base` with dropout 0.3 (the big model's value). Does it help on a dataset this small?
9. **Relative positions.** Replace the sinusoidal encoding with rotary embeddings (RoPE) applied to Q and K, and compare validation BLEU.
10. **Explain without notes**: why $\sqrt{d_k}$; why the decoder needs a causal mask; why label smoothing raises perplexity but improves BLEU; what each of the three attention types reads from.

### References

- Vaswani et al. (2017). *Attention Is All You Need.* [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- Rush (2018); Huang, Subramanian, Sum, Almubarak, Biderman (2022). *The Annotated Transformer.* [nlp.seas.harvard.edu/annotated-transformer](https://nlp.seas.harvard.edu/annotated-transformer/)
- Sennrich, Haddow, Birch (2016). *Neural Machine Translation of Rare Words with Subword Units* (BPE). [arXiv:1508.07909](https://arxiv.org/abs/1508.07909)
- Szegedy et al. (2016). *Rethinking the Inception Architecture* (label smoothing). [arXiv:1512.00567](https://arxiv.org/abs/1512.00567)
- Ba, Kiros, Hinton (2016). *Layer Normalization.* [arXiv:1607.06450](https://arxiv.org/abs/1607.06450)
- Wu et al. (2016). *Google's Neural Machine Translation System* (length penalty). [arXiv:1609.08144](https://arxiv.org/abs/1609.08144)
- Papineni et al. (2002). *BLEU: a Method for Automatic Evaluation of Machine Translation.* [ACL Anthology P02-1040](https://aclanthology.org/P02-1040/)
- Post (2018). *A Call for Clarity in Reporting BLEU Scores* (sacreBLEU). [arXiv:1804.08771](https://arxiv.org/abs/1804.08771)
- Elliott et al. (2016). *Multi30K: Multilingual English-German Image Descriptions.* [arXiv:1605.00459](https://arxiv.org/abs/1605.00459)
- Xiong et al. (2020). *On Layer Normalization in the Transformer Architecture* (pre-norm vs post-norm). [arXiv:2002.04745](https://arxiv.org/abs/2002.04745)